# 29. 최종 수치 재정리 + 순서 의존성 실험 + README 준비

## 이번 노트북에서 할 것
1. 30개 규칙 기준 최종 수치 재계산: 커버리지, 성공률, 3-endpoint 비교, QED
2. 순서 의존성 실험: 규칙기반(고정순서) vs LLM(맥락순서)의 최종 결과 차이 확인
3. GitHub README 초안 작성

## 간략한 정리 (28까지)
- 라이브러리 30개 규칙, 11가지 편집 방식, valid set 커버리지 31.5%
- 오늘 신규(27~28번): sulphate, N_oxide, 2-halo_pyridine, disulphide,
  quinone_A(370)(연쇄처리 검증), isocyanate, triple_bond, stilbene,
  beta-keto/anhydride(가수분해형, 아마이드 대안은 문헌확인 대기)
- 신규 편집 타입: reduce_multi_bond, cleave_bond, remove_atom
- MMPA 통계검정 시도 → beta-keto/anhydride는 데이터형 불가 확정,
  ChEMBL 승인약물로 Thiocarbonyl_group 실증(티오/옥소바르비투레이트 쌍)
- 3D 형태비교 지표 + 시각화 도구(src/tools/visualization.py) 완성,
  단계별 2D + 조건부 3D 자동 표시
- test set은 여전히 미사용

## 다음에 해야 할 것 (오늘 끝나면)
- phthalimide, hydroxamic_acid 등 문헌형은 학생이 논문 확인 후 진행
- 학생 승인 시 test set 1회 최종 검증
- 제안서는 학생이 계속 병행 작성 중



In [1]:
# 셀 1
!pip install rdkit -q
!pip install fuzzywuzzy python-Levenshtein -q
!pip install PyTDC --no-deps -q
!pip install PyYAML tqdm requests -q
!pip install openai -q
!pip install py3Dmol -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.4/37.4 MB 16.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 153.3/153.3 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 39.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.2/154.2 kB 4.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


In [2]:
# 셀 2
from google.colab import userdata
token = userdata.get('GITHUB_TOKEN')

!git clone https://{token}@github.com/Dec32th/laidd-2026.git
%cd /content/laidd-2026
!pwd

Cloning into 'laidd-2026'...
remote: Enumerating objects: 354, done.
remote: Counting objects: 100% (94/94), done.
remote: Compressing objects: 100% (62/62), done.
remote: Total 354 (delta 48), reused 74 (delta 32), pack-reused 260 (from 1)
Receiving objects: 100% (354/354), 972.27 KiB | 4.93 MiB/s, done.
Resolving deltas: 100% (185/185), done.
/content/laidd-2026
/content/laidd-2026


In [3]:
# 셀 3
!git config --global user.email "hyekyeong.w@gmail.com"
!git config --global user.name "Dec32th"

In [4]:
# 셀 4
import importlib, random, json
import numpy as np, pandas as pd
from collections import Counter
from scipy import stats
from rdkit import Chem
from rdkit.Chem import rdMMPA, rdFingerprintGenerator, QED, AllChem
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

import src.tools.replacement_library
import src.tools.molecule_editor
import src.tools.atom_editor
import src.tools.toxicophore_detector
import src.tools.agent
import src.tools.visualization

from src.tools.data_prep import load_tox21_clean
from src.tools.toxicophore_detector import detect_toxicophores
from src.tools.replacement_library import get_replacement_candidates
from src.tools.molecule_editor import propose_fix, canonicalize, iterative_fix_loop
from src.tools.agent import ask_llm_which_problem_to_fix, ask_llm_which_candidate_to_use
from src.tools.atom_editor import apply_atom_edit_from_rule
from src.tools.visualization import visualize_fix_process_full

data = load_tox21_clean(random_state=7)

_generator = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)
def smiles_to_ecfp(smiles):
    mol = Chem.MolFromSmiles(smiles)
    return _generator.GetFingerprintAsNumPy(mol) if mol else None

print(f"도구 로드 완료. 현재 라이브러리 규칙 수: {len(get_replacement_candidates.__globals__['REPLACEMENT_LIBRARY'])}")

[10:51:17] WARNING: not removing hydrogen atom without neighbors
[10:51:18] Explicit valence for atom # 8 Al, 6, is greater than permitted
[10:51:19] Explicit valence for atom # 3 Al, 6, is greater than permitted
[10:51:19] Explicit valence for atom # 4 Al, 6, is greater than permitted
[10:51:19] Explicit valence for atom # 4 Al, 6, is greater than permitted
[10:51:20] Explicit valence for atom # 9 Al, 6, is greater than permitted
[10:51:20] Explicit valence for atom # 5 Al, 6, is greater than permitted
[10:51:21] Explicit valence for atom # 16 Al, 6, is greater than permitted
[10:51:23] Explicit valence for atom # 20 Al, 6, is greater than permitted


전체: 7831개, 파싱 성공: 7823개, 파싱 실패(제외): 8개


[10:51:24] WARNING: not removing hydrogen atom without neighbors


도구 로드 완료. 현재 라이브러리 규칙 수: 30


In [5]:
# 셀 5 — Tox21 baseline
X_train, y_train, w_train = data['X_train'], data['y_train'], data['w_train']
task_cols = data['task_cols']
classifiers = {}
for i, task in enumerate(task_cols):
    train_mask = w_train[:, i] == 1
    clf = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)
    clf.fit(X_train[train_mask], y_train[train_mask, i])
    classifiers[task] = clf
print("Tox21 baseline 완료")

Tox21 baseline 완료


In [6]:
# 셀 6 — Ames/hERG/DILI baseline
from tdc.single_pred import Tox

def prepare_split_generic(df):
    df = df.copy()
    df['mol_valid'] = df['Drug'].apply(lambda s: Chem.MolFromSmiles(s) is not None)
    df_clean = df[df['mol_valid']].reset_index(drop=True)
    X = np.stack(df_clean['Drug'].apply(smiles_to_ecfp).values)
    y = df_clean['Y'].values
    return X, y

X_train_ames, y_train_ames = prepare_split_generic(Tox(name='AMES').get_split()['train'])
ames_clf = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)
ames_clf.fit(X_train_ames, y_train_ames)

X_train_herg, y_train_herg = prepare_split_generic(Tox(name='hERG').get_split()['train'])
herg_clf = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)
herg_clf.fit(X_train_herg, y_train_herg)

X_train_dili, y_train_dili = prepare_split_generic(Tox(name='DILI').get_split()['train'])
dili_clf = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)
dili_clf.fit(X_train_dili, y_train_dili)

def predict_ames(s):
    m = Chem.MolFromSmiles(s)
    return ames_clf.predict_proba(smiles_to_ecfp(s).reshape(1,-1))[0][1] if m else None
def predict_herg(s):
    m = Chem.MolFromSmiles(s)
    return herg_clf.predict_proba(smiles_to_ecfp(s).reshape(1,-1))[0][1] if m else None
def predict_dili(s):
    m = Chem.MolFromSmiles(s)
    return dili_clf.predict_proba(smiles_to_ecfp(s).reshape(1,-1))[0][1] if m else None
def predict_tox21_avg(s):
    m = Chem.MolFromSmiles(s)
    if not m: return None
    fp = smiles_to_ecfp(s).reshape(1,-1)
    return np.mean([classifiers[t].predict_proba(fp)[0][1] for t in task_cols])

print("Ames/hERG/DILI baseline 완료")

Downloading...
100%|██████████| 344k/344k [00:00<00:00, 1.85MiB/s]
Loading...
Done!
Downloading...
100%|██████████| 50.2k/50.2k [00:00<00:00, 839kiB/s]
Loading...
Done!
[10:52:49] WARNING: not removing hydrogen atom without neighbors
[10:52:49] WARNING: not removing hydrogen atom without neighbors
[10:52:49] WARNING: not removing hydrogen atom without neighbors
[10:52:49] WARNING: not removing hydrogen atom without neighbors
Downloading...
100%|██████████| 26.7k/26.7k [00:00<00:00, 442kiB/s]
Loading...
Done!


Ames/hERG/DILI baseline 완료


In [7]:
# 셀 7 — Qwen 연결
from openai import OpenAI
dashscope_key = userdata.get('DASHSCOPE_API_KEY')
client_qwen = OpenAI(api_key=dashscope_key, base_url="https://token-plan.ap-southeast-1.maas.aliyuncs.com/compatible-mode/v1")
print("Qwen 클라이언트 준비 완료")

Qwen 클라이언트 준비 완료


In [8]:
count_known_final5 = 0
for s in data['smiles_valid']:
    p = detect_toxicophores(s)
    known_count = sum(1 for x in p if get_replacement_candidates(x['rule_name']) is not None)
    if known_count >= 1:
        count_known_final5 += 1

print(f"Valid set 커버리지 (30개 규칙): {count_known_final5}개 / {len(data['smiles_valid'])}개 ({count_known_final5/len(data['smiles_valid'])*100:.1f}%)")

Valid set 커버리지 (30개 규칙): 384개 / 1173개 (32.7%)


In [9]:
single_known_final5 = []
multi_known_final5 = []

for s in data['smiles_valid']:
    p = detect_toxicophores(s)
    known_count = sum(1 for x in p if get_replacement_candidates(x['rule_name']) is not None)
    if known_count == 1:
        single_known_final5.append(s)
    elif known_count >= 2:
        multi_known_final5.append(s)

print(f"단일 문제 분자: {len(single_known_final5)}개")
print(f"다중 문제 분자: {len(multi_known_final5)}개")

random.seed(42)
sample_single_final5 = random.sample(single_known_final5, min(50, len(single_known_final5)))

rule_based_final5 = []
for smi in sample_single_final5:
    result = iterative_fix_loop(smi, max_iterations=10)
    rule_based_final5.append({"smiles": smi, "status": result['status'], "steps": len(result['history'])-1})

status_counts_final5 = Counter(r['status'] for r in rule_based_final5)
print("\n규칙기반 결과 (50개 표본, 30개 규칙 최종):")
for status, count in status_counts_final5.items():
    print(f"  {status}: {count}개 ({count/50*100:.1f}%)")

total_f5 = len(rule_based_final5)
success_f5 = sum(1 for r in rule_based_final5 if r['status'] == 'success')
partial_f5 = sum(1 for r in rule_based_final5 if r['status'] == 'no_known_fix' and r['steps'] >= 1)
print(f"\n최소 1단계 이상 개선: {(success_f5+partial_f5)/total_f5*100:.1f}%")

단일 문제 분자: 313개
다중 문제 분자: 71개

규칙기반 결과 (50개 표본, 30개 규칙 최종):
  success: 32개 (64.0%)
  stuck: 10개 (20.0%)
  no_known_fix: 8개 (16.0%)

최소 1단계 이상 개선: 80.0%


In [10]:
target_rules_final5 = list(get_replacement_candidates.__globals__['REPLACEMENT_LIBRARY'].keys())

verification_final5 = []
for s in data['smiles_valid']:
    problems = detect_toxicophores(s)
    known = [p for p in problems if p['rule_name'] in target_rules_final5]
    if not known:
        continue
    rule = known[0]['rule_name']
    fixed = propose_fix(s, rule, candidate_idx=0)
    if fixed is None or not fixed['is_valid']:
        continue
    verification_final5.append({"original": s, "fixed": fixed['new_smiles'], "rule": rule})
    if len(verification_final5) >= 100:
        break

print(f"검증 대상: {len(verification_final5)}개")

tox21_changes_f5, ames_changes_f5, herg_changes_f5, dili_changes_f5 = [], [], [], []
for c in verification_final5:
    ot, ft = predict_tox21_avg(c['original']), predict_tox21_avg(c['fixed'])
    oa, fa = predict_ames(c['original']), predict_ames(c['fixed'])
    oh, fh = predict_herg(c['original']), predict_herg(c['fixed'])
    od, fd = predict_dili(c['original']), predict_dili(c['fixed'])
    if None in (ot, ft, oa, fa, oh, fh, od, fd):
        continue
    tox21_changes_f5.append(ft - ot)
    ames_changes_f5.append(fa - oa)
    herg_changes_f5.append(fh - oh)
    dili_changes_f5.append(fd - od)

print(f"\n유효 비교쌍: {len(tox21_changes_f5)}개\n")
for name, changes in [("Tox21", tox21_changes_f5), ("Ames", ames_changes_f5), ("hERG", herg_changes_f5), ("DILI", dili_changes_f5)]:
    t, p = stats.ttest_1samp(changes, 0)
    improved = sum(1 for c in changes if c < 0)
    print(f"{name}: 평균변화={np.mean(changes):+.4f}, p={p:.5f}, 개선비율={improved}/{len(changes)}({improved/len(changes)*100:.1f}%)")

검증 대상: 100개

유효 비교쌍: 100개

Tox21: 평균변화=-0.0088, p=0.02847, 개선비율=67/100(67.0%)
Ames: 평균변화=-0.0862, p=0.00003, 개선비율=65/100(65.0%)
hERG: 평균변화=+0.0072, p=0.35633, 개선비율=43/100(43.0%)
DILI: 평균변화=-0.0313, p=0.00874, 개선비율=52/100(52.0%)


In [11]:
qed_changes_f5 = []
for c in verification_final5:
    orig_mol = Chem.MolFromSmiles(c['original'])
    fixed_mol = Chem.MolFromSmiles(c['fixed'])
    if orig_mol is None or fixed_mol is None:
        continue
    qed_changes_f5.append(QED.qed(fixed_mol) - QED.qed(orig_mol))

print(f"QED 변화 (n={len(qed_changes_f5)}):")
print(f"  평균 변화: {np.mean(qed_changes_f5):+.4f}")
print(f"  개선(증가) 비율: {sum(1 for x in qed_changes_f5 if x > 0)}/{len(qed_changes_f5)} ({sum(1 for x in qed_changes_f5 if x > 0)/len(qed_changes_f5)*100:.1f}%)")

QED 변화 (n=100):
  평균 변화: +0.0573
  개선(증가) 비율: 78/100 (78.0%)


In [12]:
test_frag_bug = "Cc1cc(-c2cc(C)c(N)c(C)c2)cc(C)c1N"
mol_check = Chem.MolFromSmiles(test_frag_bug)
info_aniline = get_replacement_candidates("aniline")
pattern_check = Chem.MolFromSmarts(info_aniline['problem_smarts'])
match_check = mol_check.GetSubstructMatches(pattern_check)
print("매치:", match_check)

ring_indices_check = [match_check[0][i] for i in info_aniline['candidates'][1]['ring_atom_indices_in_pattern']]
print("고리 원자:", ring_indices_check)

for ridx in ring_indices_check:
    atom = mol_check.GetAtomWithIdx(ridx)
    extra_neighbors = [n.GetIdx() for n in atom.GetNeighbors() if n.GetIdx() not in ring_indices_check]
    print(f"  idx={ridx}: 고리 밖 이웃={extra_neighbors}")

매치: ((9, 8, 6, 5, 4, 3, 12, 10), (17, 16, 14, 13, 3, 4, 2, 1))
고리 원자: [8, 6, 5, 4, 12, 10]
  idx=8: 고리 밖 이웃=[9]
  idx=6: 고리 밖 이웃=[7]
  idx=5: 고리 밖 이웃=[]
  idx=4: 고리 밖 이웃=[3]
  idx=12: 고리 밖 이웃=[]
  idx=10: 고리 밖 이웃=[11]


In [46]:
!cat src/tools/atom_editor.py

from rdkit import Chem


def apply_atom_edit_from_rule(smiles: str, rule_name: str, candidate_idx: int = 0):
    """replacement_library의 atom_edit 규칙을 이용해 원자/결합/고리 직접 편집을 수행."""
    from src.tools.replacement_library import get_replacement_candidates
    info = get_replacement_candidates(rule_name)
    if info is None or info.get("edit_method") != "atom_edit":
        return None
    if candidate_idx >= len(info["candidates"]):
        return None

    candidate = info["candidates"][candidate_idx]
    smarts = info["problem_smarts"]

    mol = Chem.MolFromSmiles(smiles)
    pattern = Chem.MolFromSmarts(smarts)
    if mol is None or pattern is None:
        return None

    matches = mol.GetSubstructMatches(pattern)
    if not matches:
        return None
    match = matches[0]

    rwmol = Chem.RWMol(mol)
    edit_type = candidate["edit_type"]

    if edit_type == "replace_element":
        target_idx = match[candidate.get("target_idx_in_pattern", info.get("target_idx_in_pattern"))]
  

In [13]:
importlib.reload(src.tools.atom_editor)
importlib.reload(src.tools.molecule_editor)
from src.tools.molecule_editor import propose_fix

result_check = propose_fix("Cc1cc(-c2cc(C)c(N)c(C)c2)cc(C)c1N", "aniline", candidate_idx=1)
print("메틸기 있는 케이스(거부되어야 함):", result_check)

result_normal = propose_fix("Cc1cc(NS(=O)(=O)c2ccc(N)cc2)nc(C)n1", "aniline", candidate_idx=1)
print("정상 케이스(성공해야 함):", result_normal)

메틸기 있는 케이스(거부되어야 함): None
정상 케이스(성공해야 함): {'new_smiles': 'Cc1cc(NS(=O)(=O)C23CC(N)(C2)C3)nc(C)n1', 'candidate_used': 'BCP (bicyclo[1.1.1]pentane)', 'rationale': 'para-이치환 아닐린의 방향족 벤젠 고리를 포화 bicyclic 탄소골격(BCP)으로 교체함. 방향족성 제거로 aniline reactive metabolite(RM) 형성 및 CYP-inhibition을 감소시켜, 퀴논이민 생성 경로를 차단하고 특이체질 약물 부작용(IADR) 위험을 낮춤 (문헌 근거, 학생 제공). 벤젠과의 공간적 유사성, Fsp3 증가, 실제 성공 사례가 많아 채택. 아마이드화(단순 아민 치환)보다 변화 폭이 크지만, 물성 개선 효과도 더 큼', 'is_valid': True}


In [18]:
!git status

On branch main
Your branch and 'origin/main' have diverged,
and have 6 and 5 different commits each, respectively.
  (use "git pull" to merge the remote branch into yours)

All conflicts fixed but you are still merging.
  (use "git commit" to conclude merge)

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	laidd-2026/



In [19]:
import os
os.environ['GIT_EDITOR'] = 'true'
!git commit --no-edit

[main c7b941b] Merge branch 'main' of https://github.com/Dec32th/laidd-2026


In [20]:
!git status
!git log --oneline -5

On branch main
Your branch is ahead of 'origin/main' by 7 commits.
  (use "git push" to publish your local commits)

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	laidd-2026/

nothing added to commit but untracked files present (use "git add" to track)
c7b941b (HEAD -> main) Merge branch 'main' of https://github.com/Dec32th/laidd-2026
aad88ac Fix critical fragmentation bug in replace_ring: ring atoms with extra substituents beyond the two anchors (e.g. methyl groups on the benzene ring) were silently orphaned during ring removal, producing disconnected fragments (e.g. 'C.C.C.C.NC12...') that were incorrectly marked is_valid=True. Added guard that rejects the substitution when unhandled ring substituents exist, plus a defense-in-depth check rejecting any result SMILES containing '.'. Discovered via order-dependency experiment where LLM's multiple BCP applications on a multi-substituted aniline produced fragmented output.
458f04d (origin/main, origin/

In [21]:
!git push origin main

Enumerating objects: 33, done.
Counting objects: 100% (32/32), done.
Delta compression using up to 2 threads
Compressing objects: 100% (15/15), done.
Writing objects: 100% (21/21), 4.67 KiB | 4.67 MiB/s, done.
Total 21 (delta 11), reused 13 (delta 6), pack-reused 0
remote: Resolving deltas: 100% (11/11), completed with 7 local objects.
To https://github.com/Dec32th/laidd-2026.git
   458f04d..c7b941b  main -> main


In [22]:
!ls laidd-2026/ 2>&1 | head -5

data
docs
models
notebooks
outputs


In [23]:
if False:
  import json

  random.seed(7)
  sample_multi_order_v2 = random.sample(multi_known_final5, min(20, len(multi_known_final5)))
  print(f"실험 표본: {len(sample_multi_order_v2)}개")

  order_comparison_v2 = []
  for i, smi in enumerate(sample_multi_order_v2):
      result_rule = iterative_fix_loop(smi, max_iterations=10)
      result_llm = iterative_fix_loop(smi, max_iterations=10,
                                      llm_client=client_qwen, llm_model="qwen3.8-max-preview",
                                      llm_client_type="openai_compatible")

      entry = {
          "original": smi,
          "rule_final": result_rule['final_smiles'],
          "rule_status": result_rule['status'],
          "rule_steps": len(result_rule['history']) - 1,
          "llm_final": result_llm['final_smiles'],
          "llm_status": result_llm['status'],
          "llm_steps": len(result_llm['history']) - 1,
      }
      order_comparison_v2.append(entry)

      # 중간 저장 (런타임 끊김 대비)
      with open("order_comparison_progress.json", "w") as f:
          json.dump(order_comparison_v2, f, ensure_ascii=False, indent=2)

      print(f"[{i+1}/{len(sample_multi_order_v2)}] 완료 - 규칙:{entry['rule_status']}({entry['rule_steps']}), LLM:{entry['llm_status']}({entry['llm_steps']})")

  print("\n전체 완료!")

실험 표본: 20개
[1/20] 완료 - 규칙:no_known_fix(2), LLM:no_known_fix(2)
[2/20] 완료 - 규칙:stuck(1), LLM:stuck(0)
[3/20] 완료 - 규칙:success(3), LLM:success(3)
[4/20] 완료 - 규칙:stuck(0), LLM:success(4)
[5/20] 완료 - 규칙:success(2), LLM:stuck(0)
[6/20] 완료 - 규칙:no_known_fix(8), LLM:no_known_fix(7)
[7/20] 완료 - 규칙:no_known_fix(2), LLM:no_known_fix(2)
[8/20] 완료 - 규칙:stuck(0), LLM:success(1)
[9/20] 완료 - 규칙:success(2), LLM:success(2)
[10/20] 완료 - 규칙:no_known_fix(3), LLM:no_known_fix(3)
[11/20] 완료 - 규칙:success(3), LLM:success(2)
[12/20] 완료 - 규칙:stuck(0), LLM:stuck(0)
[13/20] 완료 - 규칙:success(4), LLM:success(4)
[14/20] 완료 - 규칙:success(2), LLM:success(2)
[15/20] 완료 - 규칙:stuck(1), LLM:success(1)
[16/20] 완료 - 규칙:stuck(0), LLM:no_known_fix(1)
[17/20] 완료 - 규칙:stuck(0), LLM:stuck(0)
[18/20] 완료 - 규칙:no_known_fix(3), LLM:no_known_fix(3)
[19/20] 완료 - 규칙:stuck(1), LLM:stuck(1)
[20/20] 완료 - 규칙:success(2), LLM:success(2)

전체 완료!


In [18]:
stuck_originals = [r['original'] for r in order_comparison_v2
                    if r['rule_status'] == 'stuck' or r['llm_status'] == 'stuck']
print(f"stuck이 하나라도 있었던 분자: {len(stuck_originals)}개\n")

stuck_reasons = []
for smi in stuck_originals:
    result_rule_detail = iterative_fix_loop(smi, max_iterations=10)
    if result_rule_detail['status'] == 'stuck':
        print(f"[규칙기반 stuck] {smi[:50]}")
        print(f"  사유: {result_rule_detail.get('reason_detail', result_rule_detail.get('reason'))}\n")
        stuck_reasons.append(result_rule_detail.get('reason_detail', ''))

NameError: name 'order_comparison_v2' is not defined

In [48]:
%%writefile src/tools/atom_editor.py
from rdkit import Chem


def apply_atom_edit_from_rule(smiles: str, rule_name: str, candidate_idx: int = 0):
    """replacement_library의 atom_edit 규칙을 이용해 원자/결합/고리 직접 편집을 수행."""
    from src.tools.replacement_library import get_replacement_candidates
    info = get_replacement_candidates(rule_name)
    if info is None or info.get("edit_method") != "atom_edit":
        return None
    if candidate_idx >= len(info["candidates"]):
        return None

    candidate = info["candidates"][candidate_idx]
    smarts = info["problem_smarts"]

    mol = Chem.MolFromSmiles(smiles)
    pattern = Chem.MolFromSmarts(smarts)
    if mol is None or pattern is None:
        return None

    matches = mol.GetSubstructMatches(pattern)
    if not matches:
        return None
    match = matches[0]

    rwmol = Chem.RWMol(mol)
    edit_type = candidate["edit_type"]

    if edit_type == "replace_element":
        target_idx = match[candidate.get("target_idx_in_pattern", info.get("target_idx_in_pattern"))]
        atom = rwmol.GetAtomWithIdx(target_idx)
        atom.SetAtomicNum(candidate["param"])

    elif edit_type == "add_substituent":
        target_idx = match[candidate.get("target_idx_in_pattern", info.get("target_idx_in_pattern"))]
        frag = Chem.MolFromSmiles(candidate["param"])
        if frag is None:
            return None
        combined = Chem.CombineMols(rwmol.GetMol(), frag)
        rwmol = Chem.RWMol(combined)
        offset = mol.GetNumAtoms()
        rwmol.AddBond(target_idx, offset, Chem.BondType.SINGLE)
        atom = rwmol.GetAtomWithIdx(target_idx)
        if atom.GetNumExplicitHs() > 0:
            atom.SetNumExplicitHs(atom.GetNumExplicitHs() - 1)
        else:
            atom.SetNoImplicit(False)

    elif edit_type == "reduce_bond":
        pair = candidate.get("target_idx_pair_in_pattern", info.get("target_idx_pair_in_pattern"))
        idx1 = match[pair[0]]
        idx2 = match[pair[1]]
        bond = rwmol.GetBondBetweenAtoms(idx1, idx2)
        if bond is None:
            return None
        bond.SetBondType(Chem.BondType.SINGLE)
        for idx in (idx1, idx2):
            atom = rwmol.GetAtomWithIdx(idx)
            atom.SetNoImplicit(False)

    elif edit_type == "reduce_multi_bond":
        pairs = candidate.get("target_pairs_in_pattern", info.get("target_pairs_in_pattern"))
        ring_atoms_pattern = candidate.get("ring_atoms_in_pattern", info.get("ring_atoms_in_pattern"))
        ring_bonds_pattern = candidate.get("ring_bonds_in_pattern", info.get("ring_bonds_in_pattern"))

        for pair in pairs:
            idx_c = match[pair[0]]
            idx_o = match[pair[1]]
            bond = rwmol.GetBondBetweenAtoms(idx_c, idx_o)
            if bond is None:
                return None
            bond.SetBondType(Chem.BondType.SINGLE)
            rwmol.GetAtomWithIdx(idx_o).SetNoImplicit(False)
            rwmol.GetAtomWithIdx(idx_c).SetNumExplicitHs(0)
            rwmol.GetAtomWithIdx(idx_c).SetNoImplicit(False)

        ring_indices = [match[i] for i in ring_atoms_pattern]
        for a in ring_indices:
            rwmol.GetAtomWithIdx(a).SetIsAromatic(True)

        for b1, b2 in ring_bonds_pattern:
            bidx1, bidx2 = match[b1], match[b2]
            rbond = rwmol.GetBondBetweenAtoms(bidx1, bidx2)
            if rbond is None:
                return None
            rbond.SetBondType(Chem.BondType.AROMATIC)
            rbond.SetIsAromatic(True)

    elif edit_type == "replace_multi":
        for sub in candidate["param"]:
            target_idx = match[sub["idx_in_pattern"]]
            atom = rwmol.GetAtomWithIdx(target_idx)
            atom.SetAtomicNum(sub["new_element"])
            atom.SetFormalCharge(sub.get("new_charge", 0))
            atom.SetNoImplicit(False)
            atom.SetNumExplicitHs(0)

    elif edit_type == "remove_substituent":
        remove_idx = match[candidate["remove_idx_in_pattern"]]
        upgrade_idx = match[candidate["upgrade_bond_to_idx_in_pattern"]]
        center_idx = match[candidate.get("center_idx_in_pattern", 0)]

        to_remove = set()
        visited = {center_idx}

        stack = [remove_idx]
        while stack:
            cur = stack.pop()
            if cur in visited:
                continue
            visited.add(cur)
            to_remove.add(cur)
            for n in mol.GetAtomWithIdx(cur).GetNeighbors():
                if n.GetIdx() not in visited:
                    stack.append(n.GetIdx())

        visited.add(upgrade_idx)
        upgrade_atom = mol.GetAtomWithIdx(upgrade_idx)
        for n in upgrade_atom.GetNeighbors():
            if n.GetIdx() != center_idx and n.GetIdx() not in to_remove:
                stack2 = [n.GetIdx()]
                while stack2:
                    cur2 = stack2.pop()
                    if cur2 in visited:
                        continue
                    visited.add(cur2)
                    to_remove.add(cur2)
                    for n2 in mol.GetAtomWithIdx(cur2).GetNeighbors():
                        if n2.GetIdx() not in visited:
                            stack2.append(n2.GetIdx())

        for ridx in sorted(to_remove, reverse=True):
            rwmol.RemoveAtom(ridx)

        def _adjust3(idx, removed):
            shift = sum(1 for r in removed if r < idx)
            return idx - shift

        center_new = _adjust3(center_idx, to_remove)
        upgrade_new = _adjust3(upgrade_idx, to_remove)

        bond = rwmol.GetBondBetweenAtoms(center_new, upgrade_new)
        if bond is None:
            return None
        bond.SetBondType(Chem.BondType.DOUBLE)
        rwmol.GetAtomWithIdx(center_new).SetNoImplicit(False)
        rwmol.GetAtomWithIdx(upgrade_new).SetNoImplicit(False)

    elif edit_type == "remove_atom":
        remove_idx = match[candidate["remove_idx_in_pattern"]]
        center_idx = match[candidate.get("center_idx_in_pattern", 0)]

        to_remove = set()
        visited = {center_idx}
        stack = [remove_idx]
        while stack:
            cur = stack.pop()
            if cur in visited:
                continue
            visited.add(cur)
            to_remove.add(cur)
            for n in mol.GetAtomWithIdx(cur).GetNeighbors():
                if n.GetIdx() not in visited:
                    stack.append(n.GetIdx())

        for ridx in sorted(to_remove, reverse=True):
            rwmol.RemoveAtom(ridx)

        def _adjust4(idx, removed):
            shift = sum(1 for r in removed if r < idx)
            return idx - shift

        center_new = _adjust4(center_idx, to_remove)
        rwmol.GetAtomWithIdx(center_new).SetNoImplicit(False)

    elif edit_type == "cleave_bond":
        pair = candidate["cleave_pair_in_pattern"]
        idx1 = match[pair[0]]
        idx2 = match[pair[1]]
        bond = rwmol.GetBondBetweenAtoms(idx1, idx2)
        if bond is None:
            return None
        rwmol.RemoveBond(idx1, idx2)
        for idx in (idx1, idx2):
            rwmol.GetAtomWithIdx(idx).SetNoImplicit(False)

    elif edit_type == "open_epoxide":
        pair = candidate["break_pair_in_pattern"]
        idx_o = match[pair[0]]
        idx_c_break = match[pair[1]]

        bond = rwmol.GetBondBetweenAtoms(idx_o, idx_c_break)
        if bond is None:
            return None
        rwmol.RemoveBond(idx_o, idx_c_break)

        frag = Chem.MolFromSmiles("O")
        if frag is None:
            return None
        combined = Chem.CombineMols(rwmol.GetMol(), frag)
        rwmol = Chem.RWMol(combined)
        offset = mol.GetNumAtoms()
        rwmol.AddBond(idx_c_break, offset, Chem.BondType.SINGLE)

        rwmol.GetAtomWithIdx(idx_o).SetNoImplicit(False)
        rwmol.GetAtomWithIdx(idx_c_break).SetNoImplicit(False)
        rwmol.GetAtomWithIdx(offset).SetNoImplicit(False)

    elif edit_type == "replace_ring":
        ring_key = candidate.get("ring_atom_indices_in_pattern", info.get("ring_atom_indices_in_pattern"))
        anchor_key = candidate.get("anchor_indices_in_pattern", info.get("anchor_indices_in_pattern"))
        ring_indices = [match[i] for i in ring_key]
        anchor_idx1 = match[anchor_key[0]]
        anchor_idx2 = match[anchor_key[1]]

        # 안전장치: 고리 원자가 anchor 2개 외에 다른 치환기(메틸기 등)를
        # 갖고 있으면, 그 치환기가 고아가 되어 분자가 조각나므로 치환을
        # 거부한다 (다중 BCP 치환 조각화 버그 재발 방지)
        ring_set = set(ring_indices)
        for ridx in ring_indices:
            ratom = mol.GetAtomWithIdx(ridx)
            for n in ratom.GetNeighbors():
                nidx = n.GetIdx()
                if nidx not in ring_set and nidx not in (anchor_idx1, anchor_idx2):
                    return None

        anchor1_ring_neighbor = None
        anchor2_ring_neighbor = None
        for ridx in ring_indices:
            ratom = mol.GetAtomWithIdx(ridx)
            neighbor_idxs = [n.GetIdx() for n in ratom.GetNeighbors()]
            if anchor_idx1 in neighbor_idxs:
                anchor1_ring_neighbor = ridx
            if anchor_idx2 in neighbor_idxs:
                anchor2_ring_neighbor = ridx

        if anchor1_ring_neighbor is None or anchor2_ring_neighbor is None:
            return None

        frag = Chem.MolFromSmiles(candidate["param"])
        if frag is None:
            return None

        for ridx in sorted(ring_indices, reverse=True):
            rwmol.RemoveAtom(ridx)

        def _adjust(idx, removed):
            shift = sum(1 for r in removed if r < idx)
            return idx - shift

        anchor_idx1_new = _adjust(anchor_idx1, ring_indices)
        anchor_idx2_new = _adjust(anchor_idx2, ring_indices)

        combined = Chem.CombineMols(rwmol.GetMol(), frag)
        rwmol2 = Chem.RWMol(combined)
        offset = rwmol.GetMol().GetNumAtoms()

        frag_attach1 = None
        frag_attach2 = None
        for atom in frag.GetAtoms():
            if atom.GetSymbol() == '*':
                map_num = atom.GetAtomMapNum()
                if map_num == 1:
                    frag_attach1 = atom.GetIdx() + offset
                elif map_num == 2:
                    frag_attach2 = atom.GetIdx() + offset

        if frag_attach1 is None or frag_attach2 is None:
            return None

        dummy1 = rwmol2.GetAtomWithIdx(frag_attach1)
        dummy2 = rwmol2.GetAtomWithIdx(frag_attach2)
        real_neighbor1 = dummy1.GetNeighbors()[0].GetIdx()
        real_neighbor2 = dummy2.GetNeighbors()[0].GetIdx()

        rwmol2.AddBond(anchor_idx1_new, real_neighbor1, Chem.BondType.SINGLE)
        rwmol2.AddBond(anchor_idx2_new, real_neighbor2, Chem.BondType.SINGLE)
        rwmol2.RemoveAtom(max(frag_attach1, frag_attach2))
        rwmol2.RemoveAtom(min(frag_attach1, frag_attach2))

        rwmol = rwmol2
    else:
        return None

    try:
        new_mol = rwmol.GetMol()
        Chem.SanitizeMol(new_mol)
    except Exception:
        return None

    new_smiles = Chem.MolToSmiles(new_mol)

    check_mol = Chem.MolFromSmiles(new_smiles)
    is_valid = check_mol is not None
    if is_valid:
        if edit_type != "cleave_bond" and '.' in new_smiles:
            is_valid = False
        for atom in check_mol.GetAtoms():
            if (atom.GetNoImplicit() and atom.GetFormalCharge() == 0
                    and atom.GetSymbol() in ('C', 'N', 'O')
                    and atom.GetTotalNumHs() == 0 and atom.GetDegree() < 4):
                is_valid = False
                break

    return {
        "new_smiles": new_smiles,
        "candidate_used": candidate["name"],
        "rationale": candidate["rationale"],
        "is_valid": is_valid,
    }

Overwriting src/tools/atom_editor.py


In [28]:
importlib.reload(src.tools.atom_editor)
importlib.reload(src.tools.molecule_editor)
from src.tools.molecule_editor import propose_fix

print(propose_fix("S=C(SSC(=S)N1CCCCC1)N1CCCCC1", "disulphide", candidate_idx=0))

{'new_smiles': 'S=C(S)N1CCCCC1.S=C(S)N1CCCCC1', 'candidate_used': 'two thiols (bond cleaved)', 'rationale': '[참고] 이황화결합(S-S)은 시스틴/단백질의 3차구조 형성에 필수적인 정상 생체 구조이기도 하므로, 이 결합이 약물의 구조 안정성이나 표적 결합에 관여하는 경우 본 치환이 부적절할 수 있음. || 디티오카바메이트류(티우람 등) 농약/살균제에서 흔한 반응성 이황화결합을 두 개의 티올로 분리, 산화·금속킬레이팅 반응성을 낮춤 (검증 필요)', 'is_valid': True}


In [30]:
!git add src/tools/atom_editor.py order_comparison_progress.json
!git status

On branch main
Your branch is up to date with 'origin/main'.

Changes to be committed:
  (use "git restore --staged <file>..." to unstage)
	new file:   order_comparison_progress.json
	modified:   src/tools/atom_editor.py

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	laidd-2026/



In [31]:
!ls -la laidd-2026/

total 52
drwxr-xr-x 10 root root 4096 Jul 31 01:11 .
drwxr-xr-x 11 root root 4096 Jul 31 01:33 ..
drwxr-xr-x  2 root root 4096 Jul 31 01:11 data
drwxr-xr-x  2 root root 4096 Jul 31 01:11 docs
drwxr-xr-x  8 root root 4096 Jul 31 01:11 .git
-rw-r--r--  1 root root 4728 Jul 31 01:11 .gitignore
drwxr-xr-x  3 root root 4096 Jul 31 01:11 models
drwxr-xr-x  2 root root 4096 Jul 31 01:11 notebooks
drwxr-xr-x  2 root root 4096 Jul 31 01:11 outputs
-rw-r--r--  1 root root 1240 Jul 31 01:11 README.md
drwxr-xr-x  2 root root 4096 Jul 31 01:11 references
drwxr-xr-x  3 root root 4096 Jul 31 01:11 src


In [32]:
!rm -rf laidd-2026
!git status

On branch main
Your branch is up to date with 'origin/main'.

Changes to be committed:
  (use "git restore --staged <file>..." to unstage)
	new file:   order_comparison_progress.json
	modified:   src/tools/atom_editor.py



In [33]:
!git commit -m "Fix regression: the '.' fragmentation guard added earlier to catch replace_ring bugs incorrectly also rejected cleave_bond (disulphide), which legitimately produces two separate fragments by design. Now exempt cleave_bond from the fragmentation check. Verified disulphide correctly returns two thiol fragments with is_valid=True again."
!git push origin main

[main 483942b] Fix regression: the '.' fragmentation guard added earlier to catch replace_ring bugs incorrectly also rejected cleave_bond (disulphide), which legitimately produces two separate fragments by design. Now exempt cleave_bond from the fragmentation check. Verified disulphide correctly returns two thiol fragments with is_valid=True again.
 2 files changed, 183 insertions(+), 3 deletions(-)
 create mode 100644 order_comparison_progress.json
Enumerating objects: 10, done.
Counting objects: 100% (10/10), done.
Delta compression using up to 2 threads
Compressing objects: 100% (6/6), done.
Writing objects: 100% (6/6), 1.60 KiB | 1.60 MiB/s, done.
Total 6 (delta 3), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (3/3), completed with 3 local objects.
To https://github.com/Dec32th/laidd-2026.git
   c7b941b..483942b  main -> main


In [49]:
!cat src/tools/toxicophore_detector.py

from rdkit import Chem
from rdkit.Chem import FilterCatalog

def _build_catalog():
    params = FilterCatalog.FilterCatalogParams()
    params.AddCatalog(FilterCatalog.FilterCatalogParams.FilterCatalogs.PAINS)
    params.AddCatalog(FilterCatalog.FilterCatalogParams.FilterCatalogs.BRENK)
    return FilterCatalog.FilterCatalog(params)

_catalog = _build_catalog()
_oxime_pattern = Chem.MolFromSmarts("C=N[OX2H1]")
_guanidine_pattern = Chem.MolFromSmarts("[$(C(N)(N)=N)]")

_DUPLICATE_RULE_MAP = {
    "catechol_A(92)": "catechol",
    "diazo_group": "azo_A(324)",
    "oxime_1": "imine_1_oxime",
    "chinone_1": "quinone_A(370)",
}


def _refine_imine1(mol, atom_indices):
    """imine_1은 옥심(C=N-OH), 구아니딘(N-C(=N)-N), 일반 이민(C=N-R)을
    모두 포함하는 넓은 카테고리이므로, 실제 매치 부분의 화학적 맥락을
    확인해 이름을 세분화한다."""
    if mol.HasSubstructMatch(_oxime_pattern):
        matches = mol.GetSubstructMatches(_oxime_pattern)
        for match in matches:
            if set(match) & set(atom_indices):
                return

In [50]:
%%writefile src/tools/toxicophore_detector.py

from rdkit import Chem
from rdkit.Chem import FilterCatalog

def _build_catalog():
    params = FilterCatalog.FilterCatalogParams()
    params.AddCatalog(FilterCatalog.FilterCatalogParams.FilterCatalogs.PAINS)
    params.AddCatalog(FilterCatalog.FilterCatalogParams.FilterCatalogs.BRENK)
    return FilterCatalog.FilterCatalog(params)

_catalog = _build_catalog()
_oxime_pattern = Chem.MolFromSmarts("C=N[OX2H1]")
_guanidine_pattern = Chem.MolFromSmarts("[$(C(N)(N)=N)]")

_DUPLICATE_RULE_MAP = {
    "catechol_A(92)": "catechol",
    "diazo_group": "azo_A(324)",
    "oxime_1": "imine_1_oxime",
    "chinone_1": "quinone_A(370)",
}


def _refine_imine1(mol, atom_indices):
    """imine_1은 옥심(C=N-OH), 구아니딘(N-C(=N)-N), 일반 이민(C=N-R)을
    모두 포함하는 넓은 카테고리이므로, 실제 매치 부분의 화학적 맥락을
    확인해 이름을 세분화한다."""
    if mol.HasSubstructMatch(_oxime_pattern):
        matches = mol.GetSubstructMatches(_oxime_pattern)
        for match in matches:
            if set(match) & set(atom_indices):
                return "imine_1_oxime"
    if mol.HasSubstructMatch(_guanidine_pattern):
        matches = mol.GetSubstructMatches(_guanidine_pattern)
        for match in matches:
            if set(match) & set(atom_indices):
                return "imine_1_guanidine"
    return "imine_1_general"


def detect_toxicophores(smiles: str) -> list[dict]:
    """
    분자의 SMILES를 받아, FilterCatalog(PAINS+BRENK)에 매치되는
    문제 구조(toxicophore)들을 찾아서 규칙 이름과 해당 원자 인덱스를 반환.
    imine_1은 옥심/구아니딘/일반이민 하위형으로 세분화하여 반환한다.
    aniline은 FilterCatalog의 단순 [NH2] 탐지 대신, replacement_library의
    확장된 패턴(para-치환 벤젠 포함)을 그대로 사용해 재정의한다.
    PAINS/BRENK가 동일하거나 부분적으로 겹치는 구조를 서로 다른 이름/원자
    범위로 중복 보고하는 경우(예: catechol_A(92)==catechol,
    diazo_group==azo_A(324), oxime_1이 imine_1_oxime과 원자 하나 차이로
    겹침, chinone_1==quinone_A(370)), 같은 rule_name에 원자 인덱스가
    하나라도 겹치면 중복으로 간주해 제거한다(완전히 동일한 인덱스일
    필요는 없음).
    """
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return []

    results = []

    for entry in _catalog.GetMatches(mol):
        for fm in entry.GetFilterMatches(mol):
            atom_indices = sorted(set(mol_idx for _, mol_idx in fm.atomPairs))
            rule_name = entry.GetDescription()

            if rule_name == "imine_1":
                rule_name = _refine_imine1(mol, atom_indices)
            elif rule_name == "aniline":
                continue
            elif rule_name in _DUPLICATE_RULE_MAP:
                rule_name = _DUPLICATE_RULE_MAP[rule_name]

            is_duplicate = any(
                r['rule_name'] == rule_name and set(r['atom_indices']) & set(atom_indices)
                for r in results
            )
            if is_duplicate:
                continue

            results.append({
                "rule_name": rule_name,
                "atom_indices": atom_indices,
            })

    from src.tools.replacement_library import get_replacement_candidates
    aniline_info = get_replacement_candidates("aniline")
    if aniline_info:
        aniline_pattern = Chem.MolFromSmarts(aniline_info["problem_smarts"])
        if mol.HasSubstructMatch(aniline_pattern):
            matches = mol.GetSubstructMatches(aniline_pattern)
            for match in matches:
                atom_indices = sorted(set(match))
                results.append({
                    "rule_name": "aniline",
                    "atom_indices": atom_indices,
                })

    # imine_1_general과 isocyanate가 같은 원자(누적이중결합)를 가리키는
    # 경우, 처리 가능한 isocyanate를 우선하고 imine_1_general은 제거
    isocyanate_atom_sets = [set(r['atom_indices']) for r in results if r['rule_name'] == 'isocyanate']
    if isocyanate_atom_sets:
        results = [
            r for r in results
            if not (r['rule_name'] == 'imine_1_general'
                    and any(set(r['atom_indices']) & iso_set for iso_set in isocyanate_atom_sets))
        ]

    return results

Overwriting src/tools/toxicophore_detector.py


In [14]:
importlib.reload(src.tools.toxicophore_detector)
from src.tools.toxicophore_detector import detect_toxicophores

print(detect_toxicophores("COc1cc(-c2ccc(N=C=O)c(OC)c2)ccc1N=C=O"))
print(detect_toxicophores("CN=C=O"))

[{'rule_name': 'imine_1_general', 'atom_indices': [9, 10]}, {'rule_name': 'isocyanate', 'atom_indices': [9, 10, 11]}]
[{'rule_name': 'imine_1_general', 'atom_indices': [1, 2]}, {'rule_name': 'isocyanate', 'atom_indices': [1, 2, 3]}]


In [15]:
diagnostic_cases = [
    ("thioester", "CC(C)OC(=S)[S-]"),
    ("Sulfonic_acid_2", "CCCCCCCCOS(=O)(=O)[O-]"),
    ("catechol", "O=S(=O)([O-])c1ccc(N=Nc2c(O)ccc3ccccc23)cc1"),
    ("quinone_A(370)", "Nc1ccc(N)c2c1C(=O)c1ccccc1C2=O"),
    ("Three-membered_heterocycle", "COc1cc(C=O)cc2c1[C@H](COC(N)=O)[C@]1(OC(C)=O)ON2C[C@H]1OC"),
]

for rule, smi in diagnostic_cases:
    result = propose_fix(smi, rule, candidate_idx=0)
    print(f"[{rule}] {smi[:40]}")
    print(f"  결과: {result}\n")

[thioester] CC(C)OC(=S)[S-]
  결과: None

[Sulfonic_acid_2] CCCCCCCCOS(=O)(=O)[O-]
  결과: None

[catechol] O=S(=O)([O-])c1ccc(N=Nc2c(O)ccc3ccccc23)
  결과: None

[quinone_A(370)] Nc1ccc(N)c2c1C(=O)c1ccccc1C2=O
  결과: None

[Three-membered_heterocycle] COc1cc(C=O)cc2c1[C@H](COC(N)=O)[C@]1(OC(
  결과: None



In [16]:
diagnostic_targets = [
    ("thioester", "CC(C)OC(=S)[S-]"),
    ("Sulfonic_acid_2", "CCCCCCCCOS(=O)(=O)[O-]"),
    ("catechol", "O=S(=O)([O-])c1ccc(N=Nc2c(O)ccc3ccccc23)cc1"),
    ("quinone_A(370)", "Nc1ccc(N)c2c1C(=O)c1ccccc1C2=O"),
]

for rule, smi in diagnostic_targets:
    print(f"=== {rule} ===")
    info = get_replacement_candidates(rule)
    pattern = Chem.MolFromSmarts(info['problem_smarts'])
    mol = Chem.MolFromSmiles(smi)

    print(f"  SMARTS: {info['problem_smarts']}, 패턴 크기: {pattern.GetNumAtoms()}")
    print(f"  분자 전체 매치 여부: {mol.HasSubstructMatch(pattern)}")

    real_problems = [p for p in detect_toxicophores(smi) if p['rule_name'] == rule]
    print(f"  실제 FilterCatalog 매치 원자: {real_problems}")

    if info.get('edit_method') != 'atom_edit':
        frags = rdMMPA.FragmentMol(mol, maxCuts=1, resultsAsMols=False)
        print(f"  MMPA 조각 예시(최대 5개):")
        for core, chain in list(frags)[:5]:
            print(f"    core: {core}, chain: {chain}")
    print()

=== thioester ===
  SMARTS: [SX2](C(=O)), 패턴 크기: 3
  분자 전체 매치 여부: False
  실제 FilterCatalog 매치 원자: []

=== Sulfonic_acid_2 ===
  SMARTS: S(=O)(=O)[OX2H1,OX1-], 패턴 크기: 4
  분자 전체 매치 여부: True
  실제 FilterCatalog 매치 원자: [{'rule_name': 'Sulfonic_acid_2', 'atom_indices': [9, 10, 11, 12]}]
  MMPA 조각 예시(최대 5개):
    core: , chain: C[*:1].O=S(=O)([O-])OCCCCCCC[*:1]
    core: , chain: CC[*:1].O=S(=O)([O-])OCCCCCC[*:1]
    core: , chain: CCC[*:1].O=S(=O)([O-])OCCCCC[*:1]
    core: , chain: CCCC[*:1].O=S(=O)([O-])OCCCC[*:1]
    core: , chain: CCCCC[*:1].O=S(=O)([O-])OCCC[*:1]

=== catechol ===
  SMARTS: [OX2H;$(Oc1ccccc1O)], 패턴 크기: 1
  분자 전체 매치 여부: False
  실제 FilterCatalog 매치 원자: []

=== quinone_A(370) ===
  SMARTS: O=C1C=CC(=O)C=C1, 패턴 크기: 8
  분자 전체 매치 여부: False
  실제 FilterCatalog 매치 원자: [{'rule_name': 'quinone_A(370)', 'atom_indices': [6, 7, 8, 9, 10, 15, 16, 17]}]



In [19]:
stuck_full_originals = [r['original'] for r in order_comparison_v2
                          if r['rule_status'] == 'stuck']
for smi in stuck_full_originals:
    print(repr(smi))  # repr로 정확한 문자열 그대로 확인

NameError: name 'order_comparison_v2' is not defined

In [ ]:
problems_check = detect_toxicophores("CC(C)OC(=S)[S-]")
print(problems_check)

known_check = [p for p in problems_check if get_replacement_candidates(p['rule_name']) is not None]
print("known problems:", [p['rule_name'] for p in known_check])

In [ ]:
print(propose_fix("CC(C)OC(=S)[S-]", "Thiocarbonyl_group", candidate_idx=0))

In [57]:
!cat src/tools/molecule_editor.py

from rdkit import Chem
from rdkit.Chem import rdMMPA
from src.tools.replacement_library import get_replacement_candidates


def _check_and_match(part_smiles, problem_pattern, pattern_size):
    """조각이 problem_pattern과 정확한 크기로 매치되는지 확인."""
    part_mol = Chem.MolFromSmiles(part_smiles.replace('[*:1]', 'C').replace('[*:2]', 'C'))
    if part_mol is None or not part_mol.HasSubstructMatch(problem_pattern):
        return False
    n_attachment = part_smiles.count('[*:')
    return part_mol.GetNumHeavyAtoms() - n_attachment == pattern_size


def find_core_and_target(smiles: str, rule_name: str):
    """분자에서 rule_name에 해당하는 문제구조를 담은 조각(target)과
    나머지 뼈대(core)를 찾아서 반환.
    1단계(maxCuts=1)로 단순 분리를 먼저 시도하고,
    실패하면 2단계(maxCuts=2)로 고리 인접 작용기 분리를 시도한다."""
    info = get_replacement_candidates(rule_name)
    if info is None:
        return None

    problem_pattern = Chem.MolFromSmarts(info['problem_smarts'])
    pattern_size = problem_pattern.GetNumAtoms()

    mol = Chem.MolFromSmiles(smiles)


In [58]:
%%writefile src/tools/molecule_editor.py
from rdkit import Chem
from rdkit.Chem import rdMMPA
from src.tools.replacement_library import get_replacement_candidates


def _check_and_match(part_smiles, problem_pattern, pattern_size):
    """조각이 problem_pattern과 정확한 크기로 매치되는지 확인."""
    part_mol = Chem.MolFromSmiles(part_smiles.replace('[*:1]', 'C').replace('[*:2]', 'C'))
    if part_mol is None or not part_mol.HasSubstructMatch(problem_pattern):
        return False
    n_attachment = part_smiles.count('[*:')
    return part_mol.GetNumHeavyAtoms() - n_attachment == pattern_size


def find_core_and_target(smiles: str, rule_name: str):
    """분자에서 rule_name에 해당하는 문제구조를 담은 조각(target)과
    나머지 뼈대(core)를 찾아서 반환.
    1단계(maxCuts=1)로 단순 분리를 먼저 시도하고,
    실패하면 2단계(maxCuts=2)로 고리 인접 작용기 분리를 시도한다."""
    info = get_replacement_candidates(rule_name)
    if info is None:
        return None

    problem_pattern = Chem.MolFromSmarts(info['problem_smarts'])
    pattern_size = problem_pattern.GetNumAtoms()

    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None

    # --- Case A: 단순 2조각 분리 (maxCuts=1) ---
    fragments1 = rdMMPA.FragmentMol(mol, maxCuts=1, resultsAsMols=False)
    for core, chain in fragments1:
        if core:
            continue
        parts = chain.split('.')
        if len(parts) != 2:
            continue
        for i, part in enumerate(parts):
            if _check_and_match(part, problem_pattern, pattern_size):
                return {"core": parts[1 - i], "target_removed": part}

    # --- Case B: 고리 인접 등, core가 남는 2-cut 분리 ---
    fragments2 = rdMMPA.FragmentMol(mol, maxCuts=2, resultsAsMols=False)
    for core, chain in fragments2:
        if not core:
            continue
        chain_parts = chain.split('.')
        if len(chain_parts) != 2:
            continue
        for i, part in enumerate(chain_parts):
            if not _check_and_match(part, problem_pattern, pattern_size):
                continue
            other_chain_part = chain_parts[1 - i]

            target_ap = '[*:1]' if '[*:1]' in part else ('[*:2]' if '[*:2]' in part else None)
            if target_ap is None:
                continue

            core_mol = Chem.MolFromSmiles(core)
            other_mol = Chem.MolFromSmiles(other_chain_part)
            if core_mol is None or other_mol is None:
                continue
            try:
                merged = Chem.molzip(core_mol, other_mol)
            except Exception:
                continue

            merged_smiles = Chem.MolToSmiles(merged)
            if merged_smiles.count('[*:') != 1:
                continue
            if '[*:1]' not in merged_smiles:
                merged_smiles = merged_smiles.replace('[*:2]', '[*:1]')

            return {"core": merged_smiles, "target_removed": part}

    return None


def reassemble_molecule(core_smiles: str, rule_name: str, candidate_idx: int = 0):
    info = get_replacement_candidates(rule_name)
    if info is None or candidate_idx >= len(info['candidates']):
        return None
    candidate = info['candidates'][candidate_idx]

    core_mol = Chem.MolFromSmiles(core_smiles)
    replacement_mol = Chem.MolFromSmiles(f"[*:1]{candidate['smiles']}")
    if core_mol is None or replacement_mol is None:
        return None

    try:
        combined = Chem.molzip(core_mol, replacement_mol)
        new_smiles = Chem.MolToSmiles(combined)
    except Exception:
        return None

    is_valid = Chem.MolFromSmiles(new_smiles) is not None

    return {
        "new_smiles": new_smiles,
        "candidate_used": candidate['name'],
        "rationale": candidate['rationale'],
        "is_valid": is_valid,
    }


def propose_fix(smiles: str, rule_name: str, candidate_idx: int = 0):
    """규칙의 edit_method에 따라 결합절단형(기존) 또는 원자직접편집형(신규)으로 분기."""
    info = get_replacement_candidates(rule_name)
    if info is None:
        return None

    if info.get("edit_method") == "atom_edit":
        from src.tools.atom_editor import apply_atom_edit_from_rule
        return apply_atom_edit_from_rule(smiles, rule_name, candidate_idx)

    located = find_core_and_target(smiles, rule_name)
    if located is None:
        return None
    return reassemble_molecule(located['core'], rule_name, candidate_idx)


def canonicalize(smiles: str):
    mol = Chem.MolFromSmiles(smiles)
    return Chem.MolToSmiles(mol) if mol else None


def iterative_fix_loop(smiles: str, max_iterations: int = 10, candidate_idx: int = 0,
                        llm_client=None, llm_model=None, llm_client_type="gemini"):
    """진단->치환->재평가를 반복.
    llm_client가 주어지면: 어떤 문제부터 고칠지 + 어떤 후보를 쓸지 둘 다 LLM이 판단.
    llm_client_type: "gemini" 또는 "openai_compatible".
    llm_client가 없으면: 리스트 순서 + candidate_idx 고정값 사용.

    핵심 설계: 우선순위가 가장 높은 known 규칙이 실제 실행(propose_fix)에서
    실패하면, 즉시 stuck으로 끝내지 않고 그 다음 순위의 known 규칙을
    순서대로 시도한다(모두 실패해야 stuck). 이는 "진단은 됐지만 우리
    SMARTS 재구현이 특정 맥락에서 실행에 실패하는" 경우, 같은 분자에 있는
    다른 known 문제로 우회할 수 있음을 실험적으로 확인하여 반영한 설계다.

    skipped_details/reason_detail: 연구자가 no_known_fix/stuck 사유를 바로
    확인할 수 있도록 사람이 읽을 수 있는 설명과 매치된 원자 정보를 함께 제공.
    LLM이 candidate_idx=-1(치환 보류, 참고사항으로 인해 사람 검토 필요)을
    반환하면, 해당 규칙을 이번 실행에서는 flagged_for_review로 기록하고
    known_problems 취급에서 제외하여(무한 재시도 방지) 다음 후보 규칙으로 넘어간다."""
    from src.tools.toxicophore_detector import detect_toxicophores
    from src.tools.agent import ask_llm_which_problem_to_fix, ask_llm_which_candidate_to_use

    current = canonicalize(smiles)
    seen = {current}
    history = [{"step": 0, "smiles": current}]
    skipped_rules = []
    skipped_details = []
    flagged_for_review = set()

    for step in range(1, max_iterations + 1):
        problems = detect_toxicophores(current)
        history[-1]["problems"] = problems

        if not problems:
            return {"status": "success", "final_smiles": current, "history": history,
                    "skipped_rules": skipped_rules, "skipped_details": skipped_details}

        known_problems = [p for p in problems if get_replacement_candidates(p['rule_name']) is not None
                          and p['rule_name'] not in flagged_for_review]
        unknown_problems = [p for p in problems if get_replacement_candidates(p['rule_name']) is None]

        for p in unknown_problems:
            if p['rule_name'] not in skipped_rules:
                skipped_rules.append(p['rule_name'])
                mol_cur = Chem.MolFromSmiles(current)
                matched_atoms = p['atom_indices']
                atom_symbols = [mol_cur.GetAtomWithIdx(i).GetSymbol() for i in matched_atoms] if mol_cur else []
                skipped_details.append({
                    "rule_name": p['rule_name'],
                    "reason": f"라이브러리에 등록되지 않은 규칙입니다. FilterCatalog(PAINS/BRENK)가 "
                              f"'{p['rule_name']}'로 진단했으며, 매치된 원자 인덱스는 {matched_atoms}"
                              f"(원소: {atom_symbols})입니다. 이 구조에 대한 치환 규칙을 "
                              f"replacement_library.py에 추가하면 자동으로 처리 가능합니다.",
                    "atom_indices": matched_atoms,
                })

        if not known_problems:
            return {"status": "no_known_fix", "final_smiles": current, "history": history,
                    "skipped_rules": skipped_rules, "skipped_details": skipped_details}

        if llm_client is not None:
            problem_decision = ask_llm_which_problem_to_fix(llm_client, llm_model, current, problems, client_type=llm_client_type)
            preferred_rule = problem_decision['rule_name']
            problem_reason = problem_decision.get('reason', '')
            ordered_rules = [preferred_rule] + [p['rule_name'] for p in known_problems if p['rule_name'] != preferred_rule]
        else:
            problem_reason = "규칙 기반(리스트 순서대로)"
            ordered_rules = [p['rule_name'] for p in known_problems]

        fixed = None
        target_rule = None
        candidate_reason = None
        failed_attempts = []

        for candidate_rule in ordered_rules:
            if llm_client is not None:
                candidate_decision = ask_llm_which_candidate_to_use(llm_client, llm_model, current, candidate_rule, client_type=llm_client_type)
                chosen_candidate_idx = candidate_decision['candidate_idx']
                this_candidate_reason = candidate_decision.get('reason', '')

                if chosen_candidate_idx == -1:
                    flagged_for_review.add(candidate_rule)
                    if candidate_rule not in skipped_rules:
                        skipped_rules.append(candidate_rule)
                    skipped_details.append({
                        "rule_name": candidate_rule,
                        "reason": f"LLM이 치환을 보류했습니다: {this_candidate_reason} "
                                  f"(이 분자가 [참고] 사항에 해당하는 안전한 실사용 사례와 유사하다고 "
                                  f"판단되어, 자동 치환 대신 연구자의 직접 검토를 권장합니다.)",
                        "atom_indices": next((p['atom_indices'] for p in problems if p['rule_name'] == candidate_rule), []),
                    })
                    continue
            else:
                chosen_candidate_idx = candidate_idx
                this_candidate_reason = "규칙 기반(고정 인덱스)"

            attempt = propose_fix(current, candidate_rule, chosen_candidate_idx)
            if attempt is not None and attempt.get('is_valid'):
                fixed = attempt
                target_rule = candidate_rule
                candidate_reason = this_candidate_reason
                break
            else:
                failed_attempts.append(candidate_rule)

        if fixed is None:
            reason_detail = (f"이 단계에서 known 규칙 {failed_attempts} 전부를 순서대로 시도했으나 "
                              f"모두 실행에 실패했습니다. 흔한 원인: 유기금속/무기염 등 특수 화학종, "
                              f"고리 구조와의 예상치 못한 충돌, 또는 원자가 계산 오류입니다.")
            return {"status": "stuck", "reason": f"시도한 규칙 {failed_attempts} 모두 치환 실패",
                    "reason_detail": reason_detail,
                    "final_smiles": current, "history": history,
                    "skipped_rules": skipped_rules, "skipped_details": skipped_details}

        new_current = canonicalize(fixed['new_smiles'])

        if new_current in seen:
            return {"status": "cycle_detected", "final_smiles": current, "history": history,
                    "skipped_rules": skipped_rules, "skipped_details": skipped_details}

        seen.add(new_current)
        current = new_current
        history.append({
            "step": step,
            "smiles": current,
            "fixed_rule": target_rule,
            "problem_reason": problem_reason,
            "candidate_used": fixed['candidate_used'],
            "candidate_reason": candidate_reason,
        })

    return {"status": "max_iterations_reached", "final_smiles": current, "history": history,
            "skipped_rules": skipped_rules, "skipped_details": skipped_details}

Overwriting src/tools/molecule_editor.py


In [59]:
importlib.reload(src.tools.molecule_editor)
from src.tools.molecule_editor import iterative_fix_loop

result_test_retry = iterative_fix_loop("CC(C)OC(=S)[S-]", max_iterations=10)
print("상태:", result_test_retry['status'])
for h in result_test_retry['history']:
    print(h)

print("\n=== 회귀 테스트 ===")
print(propose_fix("O=C(O)CCl", "alkyl_halide", candidate_idx=0))
result_regression = iterative_fix_loop("Oc1ccc(O)cc1", max_iterations=5)
print("하이드로퀴논 연쇄:", result_regression['status'], len(result_regression['history'])-1, "단계")

상태: stuck
{'step': 0, 'smiles': 'CC(C)OC(=S)[S-]', 'problems': [{'rule_name': 'Thiocarbonyl_group', 'atom_indices': [4, 5]}, {'rule_name': 'thiol_1', 'atom_indices': [6]}]}
{'step': 1, 'smiles': 'CC(C)OC(=O)[S-]', 'fixed_rule': 'Thiocarbonyl_group', 'problem_reason': '규칙 기반(리스트 순서대로)', 'candidate_used': 'carbonyl (O replacing S)', 'candidate_reason': '규칙 기반(고정 인덱스)', 'problems': [{'rule_name': 'thioester', 'atom_indices': [4, 5, 6]}, {'rule_name': 'thiol_1', 'atom_indices': [6]}]}

=== 회귀 테스트 ===
{'new_smiles': 'O=C(O)CO', 'candidate_used': 'hydroxyl (alcohol)', 'rationale': '[참고] 메클로르에타민, 사이클로포스파미드, 카머스틴, 클로람부실 등 알킬화 항암제는 DNA 알킬화(반응성) 자체가 세포독성 치료 메커니즘이므로, 이 계열에는 본 치환이 적절하지 않음. || 이탈기를 제거해 알킬화 반응성을 없앰, 극성은 유사하게 유지', 'is_valid': True}
하이드로퀴논 연쇄: success 1 단계


In [61]:
# 지금 규칙(디티오카바메이트, 3원자)과 이번에 나온 매치(1원자)가 같은 개념인지 확인
mol_check2 = Chem.MolFromSmiles("CC(C)OC(=O)[S-]")
for atom in mol_check2.GetAtoms():
    if atom.GetSymbol() == 'S':
        print(f"idx={atom.GetIdx()}: 전하={atom.GetFormalCharge()}, 이웃={[n.GetSymbol() for n in atom.GetNeighbors()]}")

# 원래 thiol_1 규칙 SMARTS로 이 분자가 매치되는지 (매치 안 되어야 정상 - 이미 C=S가 없으므로)
info_thiol1 = get_replacement_candidates("thiol_1")
pattern_thiol1 = Chem.MolFromSmarts(info_thiol1['problem_smarts'])
print("우리 thiol_1 패턴(C(=S)[S-]) 매치:", mol_check2.HasSubstructMatch(pattern_thiol1))

idx=6: 전하=-1, 이웃=['C']
우리 thiol_1 패턴(C(=S)[S-]) 매치: False


In [62]:
%%writefile src/tools/toxicophore_detector.py
from rdkit import Chem
from rdkit.Chem import FilterCatalog

def _build_catalog():
    params = FilterCatalog.FilterCatalogParams()
    params.AddCatalog(FilterCatalog.FilterCatalogParams.FilterCatalogs.PAINS)
    params.AddCatalog(FilterCatalog.FilterCatalogParams.FilterCatalogs.BRENK)
    return FilterCatalog.FilterCatalog(params)

_catalog = _build_catalog()
_oxime_pattern = Chem.MolFromSmarts("C=N[OX2H1]")
_guanidine_pattern = Chem.MolFromSmarts("[$(C(N)(N)=N)]")

_DUPLICATE_RULE_MAP = {
    "catechol_A(92)": "catechol",
    "diazo_group": "azo_A(324)",
    "oxime_1": "imine_1_oxime",
    "chinone_1": "quinone_A(370)",
}


def _refine_imine1(mol, atom_indices):
    """imine_1은 옥심(C=N-OH), 구아니딘(N-C(=N)-N), 일반 이민(C=N-R)을
    모두 포함하는 넓은 카테고리이므로, 실제 매치 부분의 화학적 맥락을
    확인해 이름을 세분화한다."""
    if mol.HasSubstructMatch(_oxime_pattern):
        matches = mol.GetSubstructMatches(_oxime_pattern)
        for match in matches:
            if set(match) & set(atom_indices):
                return "imine_1_oxime"
    if mol.HasSubstructMatch(_guanidine_pattern):
        matches = mol.GetSubstructMatches(_guanidine_pattern)
        for match in matches:
            if set(match) & set(atom_indices):
                return "imine_1_guanidine"
    return "imine_1_general"


def _refine_thiol1(mol, atom_indices):
    """thiol_1은 FilterCatalog 원본이 음이온 황 원자 1개만 매치하는 넓은
    규칙이므로, 그 황이 붙은 탄소의 나머지 결합을 확인해 세분화한다:
    이웃 탄소가 C=S도 가지면 디티오카바메이트, C=O를 가지면
    티오카르복실산염, 둘 다 아니면 일반형(미지원)으로 분류한다."""
    s_idx = atom_indices[0]
    s_atom = mol.GetAtomWithIdx(s_idx)
    for nbr in s_atom.GetNeighbors():
        for nbr2 in nbr.GetNeighbors():
            if nbr2.GetIdx() == s_idx:
                continue
            bond2 = mol.GetBondBetweenAtoms(nbr.GetIdx(), nbr2.GetIdx())
            if bond2 is None or bond2.GetBondTypeAsDouble() != 2.0:
                continue
            if nbr2.GetSymbol() == 'S':
                return "thiol_1_dithiocarbamate"
            if nbr2.GetSymbol() == 'O':
                return "thiol_1_thiocarboxylate"
    return "thiol_1_general"


def detect_toxicophores(smiles: str) -> list[dict]:
    """
    분자의 SMILES를 받아, FilterCatalog(PAINS+BRENK)에 매치되는
    문제 구조(toxicophore)들을 찾아서 규칙 이름과 해당 원자 인덱스를 반환.
    imine_1은 옥심/구아니딘/일반이민, thiol_1은 디티오카바메이트/
    티오카르복실산염/일반형 하위형으로 세분화하여 반환한다.
    aniline은 FilterCatalog의 단순 [NH2] 탐지 대신, replacement_library의
    확장된 패턴(para-치환 벤젠 포함)을 그대로 사용해 재정의한다.
    PAINS/BRENK가 동일하거나 부분적으로 겹치는 구조를 서로 다른 이름/원자
    범위로 중복 보고하는 경우, 같은 rule_name에 원자 인덱스가 하나라도
    겹치면 중복으로 간주해 제거한다(완전히 동일한 인덱스일 필요는 없음).
    """
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return []

    results = []

    for entry in _catalog.GetMatches(mol):
        for fm in entry.GetFilterMatches(mol):
            atom_indices = sorted(set(mol_idx for _, mol_idx in fm.atomPairs))
            rule_name = entry.GetDescription()

            if rule_name == "imine_1":
                rule_name = _refine_imine1(mol, atom_indices)
            elif rule_name == "thiol_1":
                rule_name = _refine_thiol1(mol, atom_indices)
            elif rule_name == "aniline":
                continue
            elif rule_name in _DUPLICATE_RULE_MAP:
                rule_name = _DUPLICATE_RULE_MAP[rule_name]

            is_duplicate = any(
                r['rule_name'] == rule_name and set(r['atom_indices']) & set(atom_indices)
                for r in results
            )
            if is_duplicate:
                continue

            results.append({
                "rule_name": rule_name,
                "atom_indices": atom_indices,
            })

    from src.tools.replacement_library import get_replacement_candidates
    aniline_info = get_replacement_candidates("aniline")
    if aniline_info:
        aniline_pattern = Chem.MolFromSmarts(aniline_info["problem_smarts"])
        if mol.HasSubstructMatch(aniline_pattern):
            matches = mol.GetSubstructMatches(aniline_pattern)
            for match in matches:
                atom_indices = sorted(set(match))
                results.append({
                    "rule_name": "aniline",
                    "atom_indices": atom_indices,
                })

    isocyanate_atom_sets = [set(r['atom_indices']) for r in results if r['rule_name'] == 'isocyanate']
    if isocyanate_atom_sets:
        results = [
            r for r in results
            if not (r['rule_name'] == 'imine_1_general'
                    and any(set(r['atom_indices']) & iso_set for iso_set in isocyanate_atom_sets))
        ]

    return results

Overwriting src/tools/toxicophore_detector.py


In [ ]:
importlib.reload(src.tools.replacement_library)
importlib.reload(src.tools.atom_editor)
importlib.reload(src.tools.toxicophore_detector)
importlib.reload(src.tools.molecule_editor)
from src.tools.toxicophore_detector import detect_toxicophores
from src.tools.molecule_editor import propose_fix, iterative_fix_loop

print(detect_toxicophores("CC(C)OC(=S)[S-]"))
print(detect_toxicophores("CC(C)OC(=O)[S-]"))

result_full_chain = iterative_fix_loop("CC(C)OC(=S)[S-]", max_iterations=10)
print("\n전체 연쇄:", result_full_chain['status'])
for h in result_full_chain['history']:
    print(h)

In [ ]:
print(propose_fix("CC(C)OC(=O)[S-]", "thiol_1_thiocarboxylate", candidate_idx=0))

In [ ]:
print(list(get_replacement_candidates.__globals__['REPLACEMENT_LIBRARY'].keys()))

In [ ]:
info_tc = get_replacement_candidates("thiol_1_thiocarboxylate")
pattern_tc = Chem.MolFromSmarts(info_tc['problem_smarts'])
mol_tc = Chem.MolFromSmiles("CC(C)OC(=O)[S-]")
print("매치 여부:", mol_tc.HasSubstructMatch(pattern_tc))
print("매치 위치:", mol_tc.GetSubstructMatches(pattern_tc))
print("info 전체:", info_tc)

In [68]:
!cat src/tools/replacement_library.py


REPLACEMENT_LIBRARY = {
    "nitro_group": {
        "problem_smarts": "[N+](=O)[O-]",
        "candidates": [
            {"smiles": "N", "name": "primary amine",
             "rationale": "[참고] 메트로니다졸, 니트로푸란토인, 벤즈니다졸 등 일부 "
                          "항균제/항기생충제는 니트로기의 선택적 환원 활성화 자체가 "
                          "치료 메커니즘이므로, 이런 프로드러그 설계 맥락에서는 본 "
                          "치환이 적절하지 않을 수 있음. || 극성을 유지하면서 니트로기의 "
                          "환원성 대사 중간체 생성 경로를 제거함"},
            {"smiles": "S(=O)(=O)N", "name": "sulfonamide",
             "rationale": "약물유사 골격에서 흔히 쓰이는 안정적 대체기로, 수소결합 donor/acceptor 특성을 일부 유지"},
            {"smiles": "C#N", "name": "nitrile",
             "rationale": "대사 안정성이 개선된 사례가 문헌에 다수 보고됨, 다만 극성은 다소 감소"},
        ],
    },
    "aldehyde": {
        "problem_smarts": "[CX3H1](=O)",
        "candidates": [
            {"smiles": "C(=O)N", "name": "amide",
             "rationale": "알데히드의 친전자성(단백질 부가물 형성 우려)을 제거하면서 유사한 형태 유지"},
            {"smiles": "C(O)", "name": "al

In [69]:

REPLACEMENT_LIBRARY = {
    "nitro_group": {
        "problem_smarts": "[N+](=O)[O-]",
        "candidates": [
            {"smiles": "N", "name": "primary amine",
             "rationale": "[참고] 메트로니다졸, 니트로푸란토인, 벤즈니다졸 등 일부 "
                          "항균제/항기생충제는 니트로기의 선택적 환원 활성화 자체가 "
                          "치료 메커니즘이므로, 이런 프로드러그 설계 맥락에서는 본 "
                          "치환이 적절하지 않을 수 있음. || 극성을 유지하면서 니트로기의 "
                          "환원성 대사 중간체 생성 경로를 제거함"},
            {"smiles": "S(=O)(=O)N", "name": "sulfonamide",
             "rationale": "약물유사 골격에서 흔히 쓰이는 안정적 대체기로, 수소결합 donor/acceptor 특성을 일부 유지"},
            {"smiles": "C#N", "name": "nitrile",
             "rationale": "대사 안정성이 개선된 사례가 문헌에 다수 보고됨, 다만 극성은 다소 감소"},
        ],
    },
    "aldehyde": {
        "problem_smarts": "[CX3H1](=O)",
        "candidates": [
            {"smiles": "C(=O)N", "name": "amide",
             "rationale": "알데히드의 친전자성(단백질 부가물 형성 우려)을 제거하면서 유사한 형태 유지"},
            {"smiles": "C(O)", "name": "alcohol",
             "rationale": "가장 단순한 환원형 대체, 반응성 크게 감소"},
        ],
    },
    "Michael_acceptor_1": {
        "edit_method": "atom_edit",
        "problem_smarts": "C=CC(=O)",
        "target_idx_pair_in_pattern": (0, 1),
        "candidates": [
            {"edit_type": "reduce_bond", "name": "saturated (C-C single bond)",
             "rationale": "[참고] 에타크린산처럼 시스테인 잔기와의 공유결합 자체가 "
                          "작용 메커니즘인 공유결합 억제제(covalent inhibitor) "
                          "계열에는 본 경고가 그대로 적용되지 않을 수 있음. || "
                          "알파,베타-불포화 카르보닐의 C=C 이중결합을 환원하여 "
                          "단백질 친전자성 부가반응(Michael addition, covalent "
                          "binding) 위험을 제거함"},
        ],
    },
    "acid_halide": {
        "problem_smarts": "C(=O)[F,Cl,Br,I]",
        "candidates": [
            {"smiles": "C(=O)N", "name": "amide",
             "rationale": "고반응성 아실할라이드를 안정적인 아마이드로 대체"},
            {"smiles": "C(=O)O", "name": "ester",
             "rationale": "아마이드보다 극성이 낮고 유연한 대체 옵션, 가수분해 속도 조절 가능 (검증 필요)"},
        ],
    },
    "alkyl_halide": {
        "problem_smarts": "[Cl,Br,I]",
        "candidates": [
            {"smiles": "O", "name": "hydroxyl (alcohol)",
             "rationale": "[참고] 메클로르에타민, 사이클로포스파미드, 카머스틴, "
                          "클로람부실 등 알킬화 항암제는 DNA 알킬화(반응성) 자체가 "
                          "세포독성 치료 메커니즘이므로, 이 계열에는 본 치환이 "
                          "적절하지 않음. || 이탈기를 제거해 알킬화 반응성을 없앰, "
                          "극성은 유사하게 유지"},
            {"smiles": "F", "name": "fluorine",
             "rationale": "할로겐을 유지하되 C-F 결합은 강해 이탈기로 작용하지 않음, 입체적 크기도 유사"},
        ],
    },
    "aniline": {
        "edit_method": "atom_edit",
        "problem_smarts": "[NH2]c1ccc([#6,#7,#8,#16])cc1",
        "target_idx_in_pattern": 0,
        "ring_atom_indices_in_pattern": [1, 2, 3, 4, 6, 7],
        "anchor_indices_in_pattern": (0, 5),
        "candidates": [
            {"edit_type": "add_substituent", "param": "C(=O)C",
             "target_idx_in_pattern": 0,
             "name": "acetamide (acylated amine)",
             "rationale": "[참고] 설파계 항생제(설파닐아마이드, 설파메톡사졸 등)와 "
                          "프로카인아마이드처럼 아닐린 골격이 반응성 대사가 아닌 "
                          "안정적 형태로 널리 처방되어 온 사례가 다수 있음. 이 경우 "
                          "특이체질 반응은 드물고 예측이 어려워, 본 경고를 절대적 "
                          "배제 기준이 아닌 참고 신호로 해석해야 함. || 1차 방향족 "
                          "아민을 아마이드로 아실화하여 N-hydroxylation 경로 자체를 차단"},
            {"edit_type": "replace_ring", "param": "[*:1]C12CC(C1)(C2)[*:2]",
             "ring_atom_indices_in_pattern": [1, 2, 3, 4, 6, 7],
             "anchor_indices_in_pattern": (0, 5),
             "name": "BCP (bicyclo[1.1.1]pentane)",
             "rationale": "para-이치환 아닐린의 방향족 벤젠 고리를 포화 bicyclic "
                          "탄소골격(BCP)으로 교체함. 방향족성 제거로 aniline reactive "
                          "metabolite(RM) 형성 및 CYP-inhibition을 감소시켜, 퀴논이민 "
                          "생성 경로를 차단하고 특이체질 약물 부작용(IADR) 위험을 낮춤 "
                          "(문헌 근거, 학생 제공). 벤젠과의 공간적 유사성, Fsp3 증가, "
                          "실제 성공 사례가 많아 채택. 아마이드화(단순 아민 치환)보다 "
                          "변화 폭이 크지만, 물성 개선 효과도 더 큼"},
        ],
    },
    "Sulfonic_acid_2": {
        "problem_smarts": "S(=O)(=O)[OX2H1,OX1-]",
        "candidates": [
            {"smiles": "S(=O)(=O)N", "name": "sulfonamide",
             "rationale": "[참고] 암페타민 설페이트, 사퀴나비르 메실레이트처럼 "
                          "일부 승인약물에서 설폰산/설폰산 유사기는 활성 골격이 "
                          "아니라 염(salt) 형성을 위한 카운터이온으로만 존재함. "
                          "이 경우 본 규칙이 다루는 '독성 유발 골격'과 무관하므로, "
                          "치환 대상 여부를 판단하기 전에 이 산이 활성 골격의 "
                          "일부인지 염 형성용인지 구분이 필요함. || 생리적 pH에서 "
                          "이온화 정도(전하)를 크게 낮춰 세포막 투과성을 개선함. "
                          "설폰산은 대부분 음이온 상태로 존재해 경구 흡수가 저해되는 "
                          "경우가 많으나, 설폰아마이드는 유사한 골격을 유지하면서도 "
                          "중성에 가까워 약물유사성이 개선됨"},
            {"smiles": "C(=O)O", "name": "carboxylic acid",
             "rationale": "설폰산보다 산성도가 약하고 부피가 작은 산성 bioisostere "
                          "(검증 필요)"},
        ],
    },
    "imine_1_oxime": {
        "edit_method": "atom_edit",
        "problem_smarts": "C=N[OX2H1]",
        "target_idx_pair_in_pattern": (0, 1),
        "candidates": [
            {"edit_type": "reduce_bond", "name": "amine (reduced)",
             "rationale": "옥심의 C=N 결합을 환원하여, 가수분해 시 원래의 반응성 "
                          "카르보닐(알데히드/케톤)로 되돌아갈 수 있는 대사 불안정 "
                          "경로를 제거함"},
        ],
    },
    "imine_1_general": {
        "edit_method": "atom_edit",
        "problem_smarts": "[CX3;!$(C(N)(N)=N)]=N",
        "target_idx_pair_in_pattern": (0, 1),
        "candidates": [
            {"edit_type": "reduce_bond", "name": "amine (reduced)",
             "rationale": "일반 이민(C=N-R)을 환원하여 가수분해 시 반응성 카르보닐로 "
                          "되돌아갈 수 있는 대사 불안정 경로를 제거함. 옥심 특유의 "
                          "메커니즘보다는 근거가 다소 약하며, 하위 구조별 개별 검증 필요. "
                          "구아니딘(N-C(=N)-N, 공명구조로 일반 이민과 반응성이 다름)은 "
                          "이 SMARTS에서 명시적으로 제외함"},
        ],
    },
    "catechol": {
        "edit_method": "atom_edit",
        "problem_smarts": "[OX2H;$(Oc1ccccc1O)]",
        "target_idx_in_pattern": 0,
        "candidates": [
            {"edit_type": "add_substituent", "param": "C", "name": "methoxy",
             "rationale": "[참고] 도파민, 에피네프린, 이소프로테레놀 등 카테콜아민류 "
                          "약물은 카테콜 구조 자체가 아드레날린/도파민 수용체 결합에 "
                          "필수적인 약효 골격이므로, 이 경우 본 치환은 독성 감소가 "
                          "아니라 약효 상실로 이어짐. || 인체의 COMT(catechol-O-"
                          "methyltransferase) 효소가 카테콜을 메톡시페놀로 메틸화하여 "
                          "해독하는 생리적 경로와 동일한 원리. 오르토-퀴논으로의 산화 "
                          "경로를 차단하여 세포독성/유전독성 우려를 낮춤 (학생 확인 "
                          "예정: ScienceDirect catechol overview, PMC6643002 등 참고)"},
        ],
    },
    "Thiocarbonyl_group": {
        "edit_method": "atom_edit",
        "problem_smarts": "[#6]=[#16]",
        "target_idx_in_pattern": 1,
        "candidates": [
            {"edit_type": "replace_element", "param": 8, "name": "carbonyl (O replacing S)",
             "rationale": "[참고] 티오펜탈·티아밀랄(치오바르비투레이트, C=S가 지용성 "
                          "증가로 빠른 마취효과에 기여)과 티오구아닌(퓨린 유사 항대사물, "
                          "황이 작용기전에 필수)처럼 황 원자가 약효/효력에 직접 "
                          "기여하는 경우가 있어, 이 계열에는 본 치환이 부적절할 수 "
                          "있음. || 황을 산소로 대체(티오카르보닐->카르보닐)하는 것은 "
                          "흔한 bioisostere 전략으로, 갑상선 기능 저해 등 황 함유 "
                          "작용기 특유의 대사/독성 우려를 낮춤 (검증 필요, "
                          "thiourea->urea 치환 논리와 동일 계열)"},
        ],
    },
    "thiol_2": {
        "problem_smarts": "[SX2H1]",
        "candidates": [
            {"smiles": "O", "name": "hydroxyl (alcohol)",
             "rationale": "티올의 금속 킬레이팅 및 산화(이황화물/술펜산 형성) 반응성을 "
                          "제거하면서, 극성·수소결합 특성을 유사하게 유지함"},
            {"smiles": "C(=O)N", "name": "amide",
             "rationale": "티올을 아마이드로 대체하여 반응성을 낮추면서 약물유사 골격에서 "
                          "흔히 쓰이는 안정적 작용기로 전환 (검증 필요)"},
        ],
    },
    "thiol_1_dithiocarbamate": {
        "edit_method": "atom_edit",
        "problem_smarts": "C(=S)[SX1-]",
        "candidates": [
            {"edit_type": "replace_multi",
             "param": [
                 {"idx_in_pattern": 1, "new_element": 8, "new_charge": 0},
                 {"idx_in_pattern": 2, "new_element": 7, "new_charge": 0},
             ],
             "name": "carbamate (O,N replacing S,S)",
             "rationale": "디티오카바메이트(R-O-C(=S)-S-)를 카바메이트(R-O-C(=O)-N)로 "
                          "전환. 두 황 원자를 각각 산소·질소로 교체하여 금속 킬레이팅 "
                          "능력과 효소 억제 활성(디티오카바메이트류 특유의 살충제성 "
                          "독성 기전)을 제거함 (검증 필요)"},
        ],
    },
    "thiol_1_thiocarboxylate": {
        "edit_method": "atom_edit",
        "problem_smarts": "[SX1-]C(=O)",
        "target_idx_in_pattern": 0,
        "candidates": [
            {"edit_type": "replace_element", "param": 8, "name": "carboxylate (O replacing S)",
             "rationale": "티오카르복실산 음이온(R-C(=O)-S-)의 황을 산소로 대체하여 "
                          "카르복실산염(R-C(=O)-O-)으로 전환. 황 원자의 금속 킬레이팅 "
                          "및 친핵성 반응성을 제거함 (검증 필요)"},
        ],
    },
    "het-C-het_not_in_ring": {
        "edit_method": "atom_edit",
        "problem_smarts": "[CX4](O)(O)",
        "candidates": [
            {"edit_type": "remove_substituent",
             "center_idx_in_pattern": 0,
             "remove_idx_in_pattern": 1,
             "upgrade_bond_to_idx_in_pattern": 2,
             "name": "ketone/ester (one alkoxy removed, C=O formed)",
             "rationale": "아세탈/케탈 또는 오르토에스터(탄소 하나에 알콕시기 2개 "
                          "이상)는 가수분해에 민감하여 반응성 카르보닐(케톤/알데히드)로 "
                          "쉽게 분해되며 대사 불안정성을 일으킴. 알콕시기 하나를 제거하고 "
                          "남은 산소를 카르보닐로 승격시켜, 가수분해로 어차피 도달할 "
                          "안정한 최종 형태로 미리 전환함 (검증 필요)"},
        ],
    },
    "hydroquinone": {
        "edit_method": "atom_edit",
        "problem_smarts": "[OX2H]c1ccc([OX2H,NX3H1,NX3H2])cc1",
        "target_idx_in_pattern": 0,
        "candidates": [
            {"edit_type": "add_substituent", "param": "C", "name": "methoxy",
             "rationale": "[참고] 아세트아미노펜은 정상 용량에서는 안전하며 과다복용 "
                          "시에만 위험한 용량 의존적 사례임. 본 시스템은 치료지수를 "
                          "고려하지 않으므로, 아트로핀·디곡신·와파린처럼 좁은 치료지수를 "
                          "가진 기존 약물 전반에 유사하게 적용되는 한계임. || 파라 "
                          "위치에 OH와 (OH 또는 NH)가 있는 구조(하이드로퀴논/파라-"
                          "아미노페놀 계열)는 산화되어 파라-퀴논 또는 파라-퀴논이민(예: "
                          "아세트아미노펜의 NAPQI)을 형성, 글루타치온 고갈과 단백질 "
                          "공유결합을 통한 간독성 위험이 있음"},
        ],
    },
    "azo_A(324)": {
        "edit_method": "atom_edit",
        "problem_smarts": "N=N",
        "target_idx_pair_in_pattern": (0, 1),
        "candidates": [
            {"edit_type": "reduce_bond", "name": "hydrazine (reduced)",
             "rationale": "아조기(N=N)는 체내에서 아조환원효소에 의해 환원되어 두 개의 "
                          "방향족 아민으로 분해되며, 그 중 일부(벤지딘류 등)가 발암성을 "
                          "가지는 것으로 잘 알려짐(아조 색소의 대표적 독성 메커니즘). "
                          "이중결합을 환원하여 하이드라진 형태로 전환, 완전한 아민 "
                          "분해 경로 자체를 차단함 (검증 필요: 하이드라진 자체의 "
                          "잔여 반응성은 추가 확인 필요)"},
        ],
    },
    "Three-membered_heterocycle": {
        "edit_method": "atom_edit",
        "problem_smarts": "[CX4]1[OX2][CX4]1",
        "candidates": [
            {"edit_type": "open_epoxide", "break_pair_in_pattern": (1, 2),
             "name": "vicinal diol (ring-opened)",
             "rationale": "에폭시드(3원자 고리, 옥시란)는 고리 변형(strain)으로 인해 "
                          "친핵체(DNA, 단백질)와 쉽게 반응하는 알킬화제로 작용함. "
                          "체내 에폭시드 가수분해효소(epoxide hydrolase)가 실제로 "
                          "수행하는 반응과 동일하게 고리를 열어 비시날 디올(vicinal "
                          "diol)로 전환, 반응성을 제거함"},
        ],
    },
    "diketo_group": {
        "edit_method": "atom_edit",
        "problem_smarts": "C(=O)C(=O)",
        "target_idx_pair_in_pattern": (0, 1),
        "candidates": [
            {"edit_type": "reduce_bond", "name": "alpha-hydroxy ketone (reduced)",
             "rationale": "비시날 알파-디케톤(1,2-diketone)은 반응성이 높은 친전자체로 "
                          "단백질과 부가물을 형성할 수 있으며, 흡입 시 호흡기 독성을 "
                          "일으키는 것으로 알려진 디아세틸(버터향 첨가제) 사례가 대표적임. "
                          "카르보닐 하나를 환원하여 알파-하이드록시케톤(아실로인)으로 "
                          "전환, 케토-환원효소에 의한 실제 해독 경로와 유사한 방향으로 "
                          "반응성을 낮춤 (검증 필요)"},
        ],
    },
    "thioester": {
        "edit_method": "atom_edit",
        "problem_smarts": "[SX2](C(=O))",
        "target_idx_in_pattern": 0,
        "candidates": [
            {"edit_type": "replace_element", "param": 8, "name": "ester (O replacing S)",
             "rationale": "티오에스터의 황을 산소로 대체하여 일반 에스터로 전환. "
                          "티오에스터는 일반 에스터보다 가수분해 반응성이 높고 아실화 "
                          "능력이 강해 단백질 등과 부반응 우려가 있음 (검증 필요)"},
        ],
    },
    "N-nitroso": {
        "edit_method": "atom_edit",
        "problem_smarts": "[NX2;+0;!$(N(=O)[O-])]=[OX1;+0]",
        "target_idx_pair_in_pattern": (0, 1),
        "candidates": [
            {"edit_type": "reduce_bond", "name": "N-hydroxylamine (reduced)",
             "rationale": "N-니트로소 화합물(니트로사민)은 대사 활성화(알파-수산화)를 "
                          "거쳐 강력한 알킬화 발암물질을 생성하는 것으로 잘 알려짐 "
                          "(발사르탄, 라니티딘 등 실제 의약품 불순물 리콜 사례). "
                          "N=O를 환원하여 반응성을 낮춤 (검증 필요: 완전한 해독은 "
                          "탈니트로소화가 필요하며 이는 근사적 접근)"},
        ],
    },
    "hydrazine": {
        "edit_method": "atom_edit",
        "problem_smarts": "[NX3H2][NX3H1]",
        "center_idx_in_pattern": 1,
        "candidates": [
            {"edit_type": "remove_atom",
             "remove_idx_in_pattern": 0,
             "center_idx_in_pattern": 1,
             "name": "amide/amine (terminal N removed)",
             "rationale": "하이드라진/하이드라지드(R-NH-NH2)의 말단 질소를 제거하여 "
                          "단순 아민 또는 아마이드로 되돌림. 하이드라진류는 대사 시 "
                          "반응성 디아제늄 중간체를 형성해 유전독성을 일으킬 수 있는 "
                          "것으로 알려짐. 이는 azo_A(324) 환원 시 생성되는 하이드라진 "
                          "중간체의 잔여 위험을 추가로 낮추는 후속 규칙이기도 함 "
                          "(검증 필요)"},
        ],
    },
    "sulphate": {
        "edit_method": "atom_edit",
        "problem_smarts": "[OX2][SX4](=O)(=O)[OX1,OX2H]",
        "center_idx_in_pattern": 0,
        "candidates": [
            {"edit_type": "remove_atom",
             "remove_idx_in_pattern": 1,
             "center_idx_in_pattern": 0,
             "name": "alcohol (sulfate group removed)",
             "rationale": "알킬 설페이트 에스터(R-O-SO3-)는 대사되어 반응성 있는 "
                          "설페이트 이탈기를 통한 알킬화제로 작용할 수 있음(디메틸설페이트가 "
                          "강력한 발암/독성 물질로 잘 알려진 대표 사례). 설페이트기 전체를 "
                          "제거하여 원래의 알코올로 되돌림 (검증 필요)"},
        ],
    },
    "N_oxide": {
        "edit_method": "atom_edit",
        "problem_smarts": "[n+][O-]",
        "center_idx_in_pattern": 0,
        "candidates": [
            {"edit_type": "remove_atom",
             "remove_idx_in_pattern": 1,
             "center_idx_in_pattern": 0,
             "name": "pyridine (N-oxide removed)",
             "rationale": "방향족 N-옥사이드는 산화적 대사산물이자 반응성 중간체 "
                          "생성 경로의 일부일 수 있음. 산소를 제거하여 원래의 중성 "
                          "방향족 아민(피리딘 등)으로 환원, 자연 대사에서의 환원 "
                          "경로와 유사한 방향으로 반응성을 낮춤 (검증 필요)"},
        ],
    },
    "2-halo_pyridine": {
        "edit_method": "atom_edit",
        "problem_smarts": "n:c(-[Cl,Br,I])",
        "center_idx_in_pattern": 1,
        "candidates": [
            {"edit_type": "remove_atom",
             "remove_idx_in_pattern": 2,
             "center_idx_in_pattern": 1,
             "name": "pyridine (halogen removed)",
             "rationale": "피리딘 고리 질소에 인접한 위치의 할로겐(특히 불소/염소)은 "
                          "친핵성 방향족 치환(SNAr) 반응에 취약해, 체내 친핵체(글루타치온, "
                          "단백질 시스테인 등)와 반응할 수 있음. 할로겐을 제거하고 수소로 "
                          "대체하여 이 반응성 경로를 차단함 (검증 필요)"},
        ],
    },
    "disulphide": {
        "edit_method": "atom_edit",
        "problem_smarts": "[SX2][SX2]",
        "candidates": [
            {"edit_type": "cleave_bond", "cleave_pair_in_pattern": (0, 1),
             "name": "two thiols (bond cleaved)",
             "rationale": "[참고] 이황화결합(S-S)은 시스틴/단백질의 3차구조 형성에 "
                          "필수적인 정상 생체 구조이기도 하므로, 이 결합이 약물의 "
                          "구조 안정성이나 표적 결합에 관여하는 경우 본 치환이 "
                          "부적절할 수 있음. || 디티오카바메이트류(티우람 등) 농약/"
                          "살균제에서 흔한 반응성 이황화결합을 두 개의 티올로 분리, "
                          "산화·금속킬레이팅 반응성을 낮춤 (검증 필요)"},
        ],
    },
    "quinone_A(370)": {
        "edit_method": "atom_edit",
        "problem_smarts": "O=C1C=CC(=O)C=C1",
        "target_pairs_in_pattern": [(1, 0), (4, 5)],
        "ring_atoms_in_pattern": [1, 2, 3, 4, 6, 7],
        "ring_bonds_in_pattern": [(1, 2), (2, 3), (3, 4), (4, 6), (6, 7), (7, 1)],
        "candidates": [
            {"edit_type": "reduce_multi_bond", "name": "hydroquinone (reduced, re-aromatized)",
             "rationale": "파라벤조퀴논은 산화환원 사이클(redox cycling)을 통해 활성산소종(ROS)을 "
                          "생성하고 DNA/단백질과 직접 공유결합하는 대표적 반응성 구조. 체내 "
                          "NQO1(퀴논 환원효소) 효소가 실제로 수행하는 반응과 동일하게 두 카르보닐을 "
                          "환원하고 고리를 재방향족화하여 안정적인 하이드로퀴논으로 전환. 결과물이 "
                          "다시 hydroquinone 규칙에 해당할 수 있으며, 이 경우 반복 루프가 자동으로 "
                          "메톡시페놀 등 산화에 더 안정적인 형태로 한 단계 더 개선함 (검증 필요, "
                          "안트라퀴논 등 융합고리형은 미지원)"},
        ],
    },
    "isocyanate": {
        "edit_method": "atom_edit",
        "problem_smarts": "[NX2]=[CX2]=[OX1]",
        "center_idx_in_pattern": 0,
        "candidates": [
            {"edit_type": "remove_atom",
            "remove_idx_in_pattern": 1,
            "center_idx_in_pattern": 0,
            "name": "amine (NCO hydrolyzed)",
            "rationale": "이소시아네이트(R-N=C=O)는 매우 반응성이 높은 친전자체로, "
                      "단백질/아미노기와 쉽게 부가반응을 일으켜 직업성 천식·과민증을 "
                      "유발하는 것으로 잘 알려짐(TDI, MDI 등 산업용 이소시아네이트 "
                      "사례). 체내/환경에서 실제로 일어나는 가수분해 경로(R-NCO + H2O "
                      "-> R-NH2 + CO2)와 동일하게 카르보닐 탄소와 산소를 제거하고 "
                      "질소만 남겨 아민으로 전환 (검증 필요)"},
        ],
    },
    "triple_bond": {
        "problem_smarts": "C#C",
        "edit_method": "atom_edit",
        "target_idx_pair_in_pattern": (0, 1),
        "candidates": [
            {"edit_type": "reduce_bond", "name": "alkene (partially reduced)",
            "rationale": "말단 알카인(삼중결합)은 CYP450 효소에 의해 기계기반 억제"
                      "(mechanism-based inhibition) 경로로 대사되며, 반응성 케텐/"
                      "에폭사이드 중간체를 형성해 효소를 비가역적으로 불활성화할 "
                      "수 있음(에티닐에스트라디올 등에서 알려진 메커니즘). 삼중결합을 "
                      "이중결합으로 환원하여 반응성을 낮춤 (검증 필요, 완전 포화가 "
                      "아닌 부분 환원)"},
        ],
    },
    "stilbene": {
        "problem_smarts": "c-[CX3]=[CX3]-c",
        "edit_method": "atom_edit",
        "target_idx_pair_in_pattern": (1, 2),
        "candidates": [
            {"edit_type": "reduce_bond", "name": "diarylethane (reduced)",
            "rationale": "스틸벤 구조(두 방향족 고리를 잇는 C=C)는 디에틸스틸베스트롤"
                      "(DES)처럼 내분비교란 및 대사 산화를 통한 반응성 중간체 형성이 "
                      "알려진 골격. 이중결합을 환원하여 평면성을 낮추고 대사 반응성을 "
                      "완화함 (검증 필요, 에스트로겐 수용체 결합에 필요한 형태 자체를 "
                      "훼손할 수 있어 신중한 해석 필요)"},
        ],
    },
    "beta-keto/anhydride": {
        "edit_method": "atom_edit",
        "problem_smarts": "C(=O)OC(=O)",
        "center_idx_in_pattern": 2,
        "candidates": [
            {"edit_type": "remove_atom",
            "remove_idx_in_pattern": 3,
            "center_idx_in_pattern": 2,
            "name": "carboxylic acid (anhydride hydrolyzed)",
            "rationale": "산 무수물(R-C(=O)-O-C(=O)-R')은 강한 아실화제로 단백질 아미노산 "
                      "잔기와 쉽게 반응하며, 수용액 환경에서 자발적으로 가수분해되어 "
                      "두 개의 카르복실산으로 분해되는 것이 자연스러운 무독화 경로임. "
                      "한쪽 아실기를 제거하여 이 가수분해 최종형(카르복실산)으로 직접 "
                      "전환 (검증 필요). ※ 대안 후보(무수물->아마이드/이미드 bioisostere) "
                      "는 문헌 확인 후 추가 예정"},
        ],
    },
}

def get_replacement_candidates(rule_name: str) -> dict | None:
    """rule_name에 해당하는 치환 정보(SMARTS + 후보 리스트)를 반환. 없으면 None."""
    return REPLACEMENT_LIBRARY.get(rule_name)


In [70]:
importlib.reload(src.tools.replacement_library)
importlib.reload(src.tools.atom_editor)
importlib.reload(src.tools.molecule_editor)
from src.tools.molecule_editor import propose_fix, iterative_fix_loop

print(list(get_replacement_candidates.__globals__['REPLACEMENT_LIBRARY'].keys()))

result_full_chain2 = iterative_fix_loop("CC(C)OC(=S)[S-]", max_iterations=10)
print("\n전체 연쇄:", result_full_chain2['status'])
for h in result_full_chain2['history']:
    print(h)

['nitro_group', 'aldehyde', 'Michael_acceptor_1', 'acid_halide', 'alkyl_halide', 'aniline', 'Sulfonic_acid_2', 'imine_1_oxime', 'imine_1_general', 'catechol', 'Thiocarbonyl_group', 'thiol_2', 'thiol_1_dithiocarbamate', 'thiol_1_thiocarboxylate', 'het-C-het_not_in_ring', 'hydroquinone', 'azo_A(324)', 'Three-membered_heterocycle', 'diketo_group', 'thioester', 'N-nitroso', 'hydrazine', 'sulphate', 'N_oxide', '2-halo_pyridine', 'disulphide', 'quinone_A(370)', 'isocyanate', 'triple_bond', 'stilbene', 'beta-keto/anhydride']

전체 연쇄: stuck
{'step': 0, 'smiles': 'CC(C)OC(=S)[S-]', 'problems': [{'rule_name': 'Thiocarbonyl_group', 'atom_indices': [4, 5]}, {'rule_name': 'thiol_1_dithiocarbamate', 'atom_indices': [6]}]}
{'step': 1, 'smiles': 'CC(C)OC(=O)[S-]', 'fixed_rule': 'Thiocarbonyl_group', 'problem_reason': '규칙 기반(리스트 순서대로)', 'candidate_used': 'carbonyl (O replacing S)', 'candidate_reason': '규칙 기반(고정 인덱스)', 'problems': [{'rule_name': 'thioester', 'atom_indices': [4, 5, 6]}, {'rule_name': 'thi

In [71]:
print(propose_fix("CC(C)OC(=O)[S-]", "thiol_1_thiocarboxylate", candidate_idx=0))

None


In [72]:
info_tc2 = get_replacement_candidates("thiol_1_thiocarboxylate")
print("info:", info_tc2)

pattern_tc2 = Chem.MolFromSmarts(info_tc2['problem_smarts'])
mol_tc2 = Chem.MolFromSmiles("CC(C)OC(=O)[S-]")
print("패턴:", info_tc2['problem_smarts'])
print("매치 여부:", mol_tc2.HasSubstructMatch(pattern_tc2))
print("매치 위치:", mol_tc2.GetSubstructMatches(pattern_tc2))

info: {'edit_method': 'atom_edit', 'problem_smarts': '[SX1-]C(=O)', 'target_idx_in_pattern': 0, 'candidates': [{'edit_type': 'replace_element', 'param': 8, 'name': 'carboxylate (O replacing S)', 'rationale': '티오카르복실산 음이온(R-C(=O)-S-)의 황을 산소로 대체하여 카르복실산염(R-C(=O)-O-)으로 전환. 황 원자의 금속 킬레이팅 및 친핵성 반응성을 제거함 (검증 필요)'}]}
패턴: [SX1-]C(=O)
매치 여부: True
매치 위치: ((6, 4, 5),)


In [ ]:
mol_debug = Chem.MolFromSmiles("CC(C)OC(=O)[S-]")
rwmol_debug = Chem.RWMol(mol_debug)

# target_idx_in_pattern=0 -> match[0]=6 (황 원자)
atom_debug = rwmol_debug.GetAtomWithIdx(6)
print("치환 전 - 원소:", atom_debug.GetSymbol(), "전하:", atom_debug.GetFormalCharge())

atom_debug.SetAtomicNum(8)
print("치환 후 - 원소:", atom_debug.GetSymbol(), "전하:", atom_debug.GetFormalCharge())

try:
    new_mol_debug = rwmol_debug.GetMol()
    Chem.SanitizeMol(new_mol_debug)
    print("성공:", Chem.MolToSmiles(new_mol_debug))
except Exception as e:
    print("실패:", repr(e))

In [20]:
from src.tools.atom_editor import apply_atom_edit_from_rule
result_direct = apply_atom_edit_from_rule("CC(C)OC(=O)[S-]", "thiol_1_thiocarboxylate", 0)
print(result_direct)

None


In [21]:
from src.tools.replacement_library import get_replacement_candidates as grc_fresh

smiles_debug = "CC(C)OC(=O)[S-]"
rule_name_debug = "thiol_1_thiocarboxylate"
candidate_idx_debug = 0

info = grc_fresh(rule_name_debug)
print("1) info:", info)
if info is None or info.get("edit_method") != "atom_edit":
    print("   -> 여기서 None 반환됨 (edit_method 문제)")
else:
    print("   -> 통과")

if info is not None and candidate_idx_debug >= len(info["candidates"]):
    print("2) -> 여기서 None 반환됨 (candidate_idx 범위 초과)")
else:
    print("2) -> 통과, candidates 개수:", len(info["candidates"]) if info else "N/A")

candidate = info["candidates"][candidate_idx_debug]
smarts = info["problem_smarts"]
print("3) candidate:", candidate)
print("   smarts:", smarts)

mol = Chem.MolFromSmiles(smiles_debug)
pattern = Chem.MolFromSmarts(smarts)
print("4) mol is None:", mol is None, "/ pattern is None:", pattern is None)

matches = mol.GetSubstructMatches(pattern)
print("5) matches:", matches)
if not matches:
    print("   -> 여기서 None 반환됨 (매치 없음)")
else:
    print("   -> 통과")

match = matches[0]
print("6) match:", match)

rwmol = Chem.RWMol(mol)
edit_type = candidate["edit_type"]
print("7) edit_type:", edit_type)

target_idx = match[candidate.get("target_idx_in_pattern", info.get("target_idx_in_pattern"))]
print("8) target_idx:", target_idx)

atom = rwmol.GetAtomWithIdx(target_idx)
atom.SetAtomicNum(candidate["param"])
print("9) 원소 치환 완료")

try:
    new_mol = rwmol.GetMol()
    Chem.SanitizeMol(new_mol)
    print("10) SanitizeMol 성공:", Chem.MolToSmiles(new_mol))
except Exception as e:
    print("10) -> 여기서 None 반환됨 (SanitizeMol 실패):", repr(e))

1) info: None
   -> 여기서 None 반환됨 (edit_method 문제)
2) -> 통과, candidates 개수: N/A


TypeError: 'NoneType' object is not subscriptable

In [22]:
keys = list(get_replacement_candidates.__globals__['REPLACEMENT_LIBRARY'].keys())
target = "thiol_1_thiocarboxylate"
print("target in keys:", target in keys)
for k in keys:
    if 'thiol' in k:
        print(repr(k))

target in keys: False
'thiol_2'
'thiol_1'


In [24]:
%%writefile src/tools/replacement_library.py

REPLACEMENT_LIBRARY = {
    "nitro_group": {
        "problem_smarts": "[N+](=O)[O-]",
        "candidates": [
            {"smiles": "N", "name": "primary amine",
             "rationale": "[참고] 메트로니다졸, 니트로푸란토인, 벤즈니다졸 등 일부 "
                          "항균제/항기생충제는 니트로기의 선택적 환원 활성화 자체가 "
                          "치료 메커니즘이므로, 이런 프로드러그 설계 맥락에서는 본 "
                          "치환이 적절하지 않을 수 있음. || 극성을 유지하면서 니트로기의 "
                          "환원성 대사 중간체 생성 경로를 제거함"},
            {"smiles": "S(=O)(=O)N", "name": "sulfonamide",
             "rationale": "약물유사 골격에서 흔히 쓰이는 안정적 대체기로, 수소결합 donor/acceptor 특성을 일부 유지"},
            {"smiles": "C#N", "name": "nitrile",
             "rationale": "대사 안정성이 개선된 사례가 문헌에 다수 보고됨, 다만 극성은 다소 감소"},
        ],
    },
    "aldehyde": {
        "problem_smarts": "[CX3H1](=O)",
        "candidates": [
            {"smiles": "C(=O)N", "name": "amide",
             "rationale": "알데히드의 친전자성(단백질 부가물 형성 우려)을 제거하면서 유사한 형태 유지"},
            {"smiles": "C(O)", "name": "alcohol",
             "rationale": "가장 단순한 환원형 대체, 반응성 크게 감소"},
        ],
    },
    "Michael_acceptor_1": {
        "edit_method": "atom_edit",
        "problem_smarts": "C=CC(=O)",
        "target_idx_pair_in_pattern": (0, 1),
        "candidates": [
            {"edit_type": "reduce_bond", "name": "saturated (C-C single bond)",
             "rationale": "[참고] 에타크린산처럼 시스테인 잔기와의 공유결합 자체가 "
                          "작용 메커니즘인 공유결합 억제제(covalent inhibitor) "
                          "계열에는 본 경고가 그대로 적용되지 않을 수 있음. || "
                          "알파,베타-불포화 카르보닐의 C=C 이중결합을 환원하여 "
                          "단백질 친전자성 부가반응(Michael addition, covalent "
                          "binding) 위험을 제거함"},
        ],
    },
    "acid_halide": {
        "problem_smarts": "C(=O)[F,Cl,Br,I]",
        "candidates": [
            {"smiles": "C(=O)N", "name": "amide",
             "rationale": "고반응성 아실할라이드를 안정적인 아마이드로 대체"},
            {"smiles": "C(=O)O", "name": "ester",
             "rationale": "아마이드보다 극성이 낮고 유연한 대체 옵션, 가수분해 속도 조절 가능 (검증 필요)"},
        ],
    },
    "alkyl_halide": {
        "problem_smarts": "[Cl,Br,I]",
        "candidates": [
            {"smiles": "O", "name": "hydroxyl (alcohol)",
             "rationale": "[참고] 메클로르에타민, 사이클로포스파미드, 카머스틴, "
                          "클로람부실 등 알킬화 항암제는 DNA 알킬화(반응성) 자체가 "
                          "세포독성 치료 메커니즘이므로, 이 계열에는 본 치환이 "
                          "적절하지 않음. || 이탈기를 제거해 알킬화 반응성을 없앰, "
                          "극성은 유사하게 유지"},
            {"smiles": "F", "name": "fluorine",
             "rationale": "할로겐을 유지하되 C-F 결합은 강해 이탈기로 작용하지 않음, 입체적 크기도 유사"},
        ],
    },
    "aniline": {
        "edit_method": "atom_edit",
        "problem_smarts": "[NH2]c1ccc([#6,#7,#8,#16])cc1",
        "target_idx_in_pattern": 0,
        "ring_atom_indices_in_pattern": [1, 2, 3, 4, 6, 7],
        "anchor_indices_in_pattern": (0, 5),
        "candidates": [
            {"edit_type": "add_substituent", "param": "C(=O)C",
             "target_idx_in_pattern": 0,
             "name": "acetamide (acylated amine)",
             "rationale": "[참고] 설파계 항생제(설파닐아마이드, 설파메톡사졸 등)와 "
                          "프로카인아마이드처럼 아닐린 골격이 반응성 대사가 아닌 "
                          "안정적 형태로 널리 처방되어 온 사례가 다수 있음. 이 경우 "
                          "특이체질 반응은 드물고 예측이 어려워, 본 경고를 절대적 "
                          "배제 기준이 아닌 참고 신호로 해석해야 함. || 1차 방향족 "
                          "아민을 아마이드로 아실화하여 N-hydroxylation 경로 자체를 차단"},
            {"edit_type": "replace_ring", "param": "[*:1]C12CC(C1)(C2)[*:2]",
             "ring_atom_indices_in_pattern": [1, 2, 3, 4, 6, 7],
             "anchor_indices_in_pattern": (0, 5),
             "name": "BCP (bicyclo[1.1.1]pentane)",
             "rationale": "para-이치환 아닐린의 방향족 벤젠 고리를 포화 bicyclic "
                          "탄소골격(BCP)으로 교체함. 방향족성 제거로 aniline reactive "
                          "metabolite(RM) 형성 및 CYP-inhibition을 감소시켜, 퀴논이민 "
                          "생성 경로를 차단하고 특이체질 약물 부작용(IADR) 위험을 낮춤 "
                          "(문헌 근거, 학생 제공). 벤젠과의 공간적 유사성, Fsp3 증가, "
                          "실제 성공 사례가 많아 채택. 아마이드화(단순 아민 치환)보다 "
                          "변화 폭이 크지만, 물성 개선 효과도 더 큼"},
        ],
    },
    "Sulfonic_acid_2": {
        "problem_smarts": "[#6]S(=O)(=O)[OX2H1,OX1-]",
        "candidates": [
            {"smiles": "S(=O)(=O)N", "name": "sulfonamide",
             "rationale": "[참고] 암페타민 설페이트, 사퀴나비르 메실레이트처럼 "
                          "일부 승인약물에서 설폰산/설폰산 유사기는 활성 골격이 "
                          "아니라 염(salt) 형성을 위한 카운터이온으로만 존재함. "
                          "이 경우 본 규칙이 다루는 '독성 유발 골격'과 무관하므로, "
                          "치환 대상 여부를 판단하기 전에 이 산이 활성 골격의 "
                          "일부인지 염 형성용인지 구분이 필요함. || 생리적 pH에서 "
                          "이온화 정도(전하)를 크게 낮춰 세포막 투과성을 개선함. "
                          "설폰산은 대부분 음이온 상태로 존재해 경구 흡수가 저해되는 "
                          "경우가 많으나, 설폰아마이드는 유사한 골격을 유지하면서도 "
                          "중성에 가까워 약물유사성이 개선됨"},
            {"smiles": "C(=O)O", "name": "carboxylic acid",
             "rationale": "설폰산보다 산성도가 약하고 부피가 작은 산성 bioisostere "
                          "(검증 필요)"},
        ],
    },
    "imine_1_oxime": {
        "edit_method": "atom_edit",
        "problem_smarts": "C=N[OX2H1]",
        "target_idx_pair_in_pattern": (0, 1),
        "candidates": [
            {"edit_type": "reduce_bond", "name": "amine (reduced)",
             "rationale": "옥심의 C=N 결합을 환원하여, 가수분해 시 원래의 반응성 "
                          "카르보닐(알데히드/케톤)로 되돌아갈 수 있는 대사 불안정 "
                          "경로를 제거함"},
        ],
    },
    "imine_1_general": {
        "edit_method": "atom_edit",
        "problem_smarts": "[CX3;!$(C(N)(N)=N)]=N",
        "target_idx_pair_in_pattern": (0, 1),
        "candidates": [
            {"edit_type": "reduce_bond", "name": "amine (reduced)",
             "rationale": "일반 이민(C=N-R)을 환원하여 가수분해 시 반응성 카르보닐로 "
                          "되돌아갈 수 있는 대사 불안정 경로를 제거함. 옥심 특유의 "
                          "메커니즘보다는 근거가 다소 약하며, 하위 구조별 개별 검증 필요. "
                          "구아니딘(N-C(=N)-N, 공명구조로 일반 이민과 반응성이 다름)은 "
                          "이 SMARTS에서 명시적으로 제외함"},
        ],
    },
    "catechol": {
        "edit_method": "atom_edit",
        "problem_smarts": "[OX2H;$(Oc1ccccc1O)]",
        "target_idx_in_pattern": 0,
        "candidates": [
            {"edit_type": "add_substituent", "param": "C", "name": "methoxy",
             "rationale": "[참고] 도파민, 에피네프린, 이소프로테레놀 등 카테콜아민류 "
                          "약물은 카테콜 구조 자체가 아드레날린/도파민 수용체 결합에 "
                          "필수적인 약효 골격이므로, 이 경우 본 치환은 독성 감소가 "
                          "아니라 약효 상실로 이어짐. || 인체의 COMT(catechol-O-"
                          "methyltransferase) 효소가 카테콜을 메톡시페놀로 메틸화하여 "
                          "해독하는 생리적 경로와 동일한 원리. 오르토-퀴논으로의 산화 "
                          "경로를 차단하여 세포독성/유전독성 우려를 낮춤 (학생 확인 "
                          "예정: ScienceDirect catechol overview, PMC6643002 등 참고)"},
        ],
    },
    "Thiocarbonyl_group": {
        "edit_method": "atom_edit",
        "problem_smarts": "[#6]=[#16]",
        "target_idx_in_pattern": 1,
        "candidates": [
            {"edit_type": "replace_element", "param": 8, "name": "carbonyl (O replacing S)",
             "rationale": "[참고] 티오펜탈·티아밀랄(치오바르비투레이트, C=S가 지용성 "
                          "증가로 빠른 마취효과에 기여)과 티오구아닌(퓨린 유사 항대사물, "
                          "황이 작용기전에 필수)처럼 황 원자가 약효/효력에 직접 "
                          "기여하는 경우가 있어, 이 계열에는 본 치환이 부적절할 수 "
                          "있음. || 황을 산소로 대체(티오카르보닐->카르보닐)하는 것은 "
                          "흔한 bioisostere 전략으로, 갑상선 기능 저해 등 황 함유 "
                          "작용기 특유의 대사/독성 우려를 낮춤 (검증 필요, "
                          "thiourea->urea 치환 논리와 동일 계열)"},
        ],
    },
    "thiol_2": {
        "problem_smarts": "[SX2H1]",
        "candidates": [
            {"smiles": "O", "name": "hydroxyl (alcohol)",
             "rationale": "티올의 금속 킬레이팅 및 산화(이황화물/술펜산 형성) 반응성을 "
                          "제거하면서, 극성·수소결합 특성을 유사하게 유지함"},
            {"smiles": "C(=O)N", "name": "amide",
             "rationale": "티올을 아마이드로 대체하여 반응성을 낮추면서 약물유사 골격에서 "
                          "흔히 쓰이는 안정적 작용기로 전환 (검증 필요)"},
        ],
    },
    "thiol_1_dithiocarbamate": {
        "edit_method": "atom_edit",
        "problem_smarts": "C(=S)[SX1-]",
        "candidates": [
            {"edit_type": "replace_multi",
             "param": [
                 {"idx_in_pattern": 1, "new_element": 8, "new_charge": 0},
                 {"idx_in_pattern": 2, "new_element": 7, "new_charge": 0},
             ],
             "name": "carbamate (O,N replacing S,S)",
             "rationale": "디티오카바메이트(R-O-C(=S)-S-)를 카바메이트(R-O-C(=O)-N)로 "
                          "전환. 두 황 원자를 각각 산소·질소로 교체하여 금속 킬레이팅 "
                          "능력과 효소 억제 활성(디티오카바메이트류 특유의 살충제성 "
                          "독성 기전)을 제거함 (검증 필요)"},
        ],
    },
    "thiol_1_thiocarboxylate": {
        "edit_method": "atom_edit",
        "problem_smarts": "[SX1-]C(=O)",
        "target_idx_in_pattern": 0,
        "candidates": [
            {"edit_type": "replace_element", "param": 8, "name": "carboxylate (O replacing S)",
             "rationale": "티오카르복실산 음이온(R-C(=O)-S-)의 황을 산소로 대체하여 "
                          "카르복실산염(R-C(=O)-O-)으로 전환. 황 원자의 금속 킬레이팅 "
                          "및 친핵성 반응성을 제거함 (검증 필요)"},
        ],
    },
    "het-C-het_not_in_ring": {
        "edit_method": "atom_edit",
        "problem_smarts": "[CX4](O)(O)",
        "candidates": [
            {"edit_type": "remove_substituent",
             "center_idx_in_pattern": 0,
             "remove_idx_in_pattern": 1,
             "upgrade_bond_to_idx_in_pattern": 2,
             "name": "ketone/ester (one alkoxy removed, C=O formed)",
             "rationale": "아세탈/케탈 또는 오르토에스터(탄소 하나에 알콕시기 2개 "
                          "이상)는 가수분해에 민감하여 반응성 카르보닐(케톤/알데히드)로 "
                          "쉽게 분해되며 대사 불안정성을 일으킴. 알콕시기 하나를 제거하고 "
                          "남은 산소를 카르보닐로 승격시켜, 가수분해로 어차피 도달할 "
                          "안정한 최종 형태로 미리 전환함 (검증 필요)"},
        ],
    },
    "hydroquinone": {
        "edit_method": "atom_edit",
        "problem_smarts": "[OX2H]c1ccc([OX2H,NX3H1,NX3H2])cc1",
        "target_idx_in_pattern": 0,
        "candidates": [
            {"edit_type": "add_substituent", "param": "C", "name": "methoxy",
             "rationale": "[참고] 아세트아미노펜은 정상 용량에서는 안전하며 과다복용 "
                          "시에만 위험한 용량 의존적 사례임. 본 시스템은 치료지수를 "
                          "고려하지 않으므로, 아트로핀·디곡신·와파린처럼 좁은 치료지수를 "
                          "가진 기존 약물 전반에 유사하게 적용되는 한계임. || 파라 "
                          "위치에 OH와 (OH 또는 NH)가 있는 구조(하이드로퀴논/파라-"
                          "아미노페놀 계열)는 산화되어 파라-퀴논 또는 파라-퀴논이민(예: "
                          "아세트아미노펜의 NAPQI)을 형성, 글루타치온 고갈과 단백질 "
                          "공유결합을 통한 간독성 위험이 있음"},
        ],
    },
    "azo_A(324)": {
        "edit_method": "atom_edit",
        "problem_smarts": "N=N",
        "target_idx_pair_in_pattern": (0, 1),
        "candidates": [
            {"edit_type": "reduce_bond", "name": "hydrazine (reduced)",
             "rationale": "아조기(N=N)는 체내에서 아조환원효소에 의해 환원되어 두 개의 "
                          "방향족 아민으로 분해되며, 그 중 일부(벤지딘류 등)가 발암성을 "
                          "가지는 것으로 잘 알려짐(아조 색소의 대표적 독성 메커니즘). "
                          "이중결합을 환원하여 하이드라진 형태로 전환, 완전한 아민 "
                          "분해 경로 자체를 차단함 (검증 필요: 하이드라진 자체의 "
                          "잔여 반응성은 추가 확인 필요)"},
        ],
    },
    "Three-membered_heterocycle": {
        "edit_method": "atom_edit",
        "problem_smarts": "[CX4]1[OX2][CX4]1",
        "candidates": [
            {"edit_type": "open_epoxide", "break_pair_in_pattern": (1, 2),
             "name": "vicinal diol (ring-opened)",
             "rationale": "에폭시드(3원자 고리, 옥시란)는 고리 변형(strain)으로 인해 "
                          "친핵체(DNA, 단백질)와 쉽게 반응하는 알킬화제로 작용함. "
                          "체내 에폭시드 가수분해효소(epoxide hydrolase)가 실제로 "
                          "수행하는 반응과 동일하게 고리를 열어 비시날 디올(vicinal "
                          "diol)로 전환, 반응성을 제거함"},
        ],
    },
    "diketo_group": {
        "edit_method": "atom_edit",
        "problem_smarts": "C(=O)C(=O)",
        "target_idx_pair_in_pattern": (0, 1),
        "candidates": [
            {"edit_type": "reduce_bond", "name": "alpha-hydroxy ketone (reduced)",
             "rationale": "비시날 알파-디케톤(1,2-diketone)은 반응성이 높은 친전자체로 "
                          "단백질과 부가물을 형성할 수 있으며, 흡입 시 호흡기 독성을 "
                          "일으키는 것으로 알려진 디아세틸(버터향 첨가제) 사례가 대표적임. "
                          "카르보닐 하나를 환원하여 알파-하이드록시케톤(아실로인)으로 "
                          "전환, 케토-환원효소에 의한 실제 해독 경로와 유사한 방향으로 "
                          "반응성을 낮춤 (검증 필요)"},
        ],
    },
    "thioester": {
        "edit_method": "atom_edit",
        "problem_smarts": "[SX2](C(=O))",
        "target_idx_in_pattern": 0,
        "candidates": [
            {"edit_type": "replace_element", "param": 8, "name": "ester (O replacing S)",
             "rationale": "티오에스터의 황을 산소로 대체하여 일반 에스터로 전환. "
                          "티오에스터는 일반 에스터보다 가수분해 반응성이 높고 아실화 "
                          "능력이 강해 단백질 등과 부반응 우려가 있음 (검증 필요)"},
        ],
    },
    "N-nitroso": {
        "edit_method": "atom_edit",
        "problem_smarts": "[NX2;+0;!$(N(=O)[O-])]=[OX1;+0]",
        "target_idx_pair_in_pattern": (0, 1),
        "candidates": [
            {"edit_type": "reduce_bond", "name": "N-hydroxylamine (reduced)",
             "rationale": "N-니트로소 화합물(니트로사민)은 대사 활성화(알파-수산화)를 "
                          "거쳐 강력한 알킬화 발암물질을 생성하는 것으로 잘 알려짐 "
                          "(발사르탄, 라니티딘 등 실제 의약품 불순물 리콜 사례). "
                          "N=O를 환원하여 반응성을 낮춤 (검증 필요: 완전한 해독은 "
                          "탈니트로소화가 필요하며 이는 근사적 접근)"},
        ],
    },
    "hydrazine": {
        "edit_method": "atom_edit",
        "problem_smarts": "[NX3H2][NX3H1]",
        "center_idx_in_pattern": 1,
        "candidates": [
            {"edit_type": "remove_atom",
             "remove_idx_in_pattern": 0,
             "center_idx_in_pattern": 1,
             "name": "amide/amine (terminal N removed)",
             "rationale": "하이드라진/하이드라지드(R-NH-NH2)의 말단 질소를 제거하여 "
                          "단순 아민 또는 아마이드로 되돌림. 하이드라진류는 대사 시 "
                          "반응성 디아제늄 중간체를 형성해 유전독성을 일으킬 수 있는 "
                          "것으로 알려짐. 이는 azo_A(324) 환원 시 생성되는 하이드라진 "
                          "중간체의 잔여 위험을 추가로 낮추는 후속 규칙이기도 함 "
                          "(검증 필요)"},
        ],
    },
    "sulphate": {
        "edit_method": "atom_edit",
        "problem_smarts": "[OX2][SX4](=O)(=O)[OX1,OX2H]",
        "center_idx_in_pattern": 0,
        "candidates": [
            {"edit_type": "remove_atom",
             "remove_idx_in_pattern": 1,
             "center_idx_in_pattern": 0,
             "name": "alcohol (sulfate group removed)",
             "rationale": "알킬 설페이트 에스터(R-O-SO3-)는 대사되어 반응성 있는 "
                          "설페이트 이탈기를 통한 알킬화제로 작용할 수 있음(디메틸설페이트가 "
                          "강력한 발암/독성 물질로 잘 알려진 대표 사례). 설페이트기 전체를 "
                          "제거하여 원래의 알코올로 되돌림 (검증 필요)"},
        ],
    },
    "N_oxide": {
        "edit_method": "atom_edit",
        "problem_smarts": "[n+][O-]",
        "center_idx_in_pattern": 0,
        "candidates": [
            {"edit_type": "remove_atom",
             "remove_idx_in_pattern": 1,
             "center_idx_in_pattern": 0,
             "name": "pyridine (N-oxide removed)",
             "rationale": "방향족 N-옥사이드는 산화적 대사산물이자 반응성 중간체 "
                          "생성 경로의 일부일 수 있음. 산소를 제거하여 원래의 중성 "
                          "방향족 아민(피리딘 등)으로 환원, 자연 대사에서의 환원 "
                          "경로와 유사한 방향으로 반응성을 낮춤 (검증 필요)"},
        ],
    },
    "2-halo_pyridine": {
        "edit_method": "atom_edit",
        "problem_smarts": "n:c(-[Cl,Br,I])",
        "center_idx_in_pattern": 1,
        "candidates": [
            {"edit_type": "remove_atom",
             "remove_idx_in_pattern": 2,
             "center_idx_in_pattern": 1,
             "name": "pyridine (halogen removed)",
             "rationale": "피리딘 고리 질소에 인접한 위치의 할로겐(특히 불소/염소)은 "
                          "친핵성 방향족 치환(SNAr) 반응에 취약해, 체내 친핵체(글루타치온, "
                          "단백질 시스테인 등)와 반응할 수 있음. 할로겐을 제거하고 수소로 "
                          "대체하여 이 반응성 경로를 차단함 (검증 필요)"},
        ],
    },
    "disulphide": {
        "edit_method": "atom_edit",
        "problem_smarts": "[SX2][SX2]",
        "candidates": [
            {"edit_type": "cleave_bond", "cleave_pair_in_pattern": (0, 1),
             "name": "two thiols (bond cleaved)",
             "rationale": "[참고] 이황화결합(S-S)은 시스틴/단백질의 3차구조 형성에 "
                          "필수적인 정상 생체 구조이기도 하므로, 이 결합이 약물의 "
                          "구조 안정성이나 표적 결합에 관여하는 경우 본 치환이 "
                          "부적절할 수 있음. || 디티오카바메이트류(티우람 등) 농약/"
                          "살균제에서 흔한 반응성 이황화결합을 두 개의 티올로 분리, "
                          "산화·금속킬레이팅 반응성을 낮춤 (검증 필요)"},
        ],
    },
    "quinone_A(370)": {
        "edit_method": "atom_edit",
        "problem_smarts": "O=C1C=CC(=O)C=C1",
        "target_pairs_in_pattern": [(1, 0), (4, 5)],
        "ring_atoms_in_pattern": [1, 2, 3, 4, 6, 7],
        "ring_bonds_in_pattern": [(1, 2), (2, 3), (3, 4), (4, 6), (6, 7), (7, 1)],
        "candidates": [
            {"edit_type": "reduce_multi_bond", "name": "hydroquinone (reduced, re-aromatized)",
             "rationale": "파라벤조퀴논은 산화환원 사이클(redox cycling)을 통해 활성산소종(ROS)을 "
                          "생성하고 DNA/단백질과 직접 공유결합하는 대표적 반응성 구조. 체내 "
                          "NQO1(퀴논 환원효소) 효소가 실제로 수행하는 반응과 동일하게 두 카르보닐을 "
                          "환원하고 고리를 재방향족화하여 안정적인 하이드로퀴논으로 전환. 결과물이 "
                          "다시 hydroquinone 규칙에 해당할 수 있으며, 이 경우 반복 루프가 자동으로 "
                          "메톡시페놀 등 산화에 더 안정적인 형태로 한 단계 더 개선함 (검증 필요, "
                          "안트라퀴논 등 융합고리형은 미지원)"},
        ],
    },
    "isocyanate": {
        "edit_method": "atom_edit",
        "problem_smarts": "[NX2]=[CX2]=[OX1]",
        "center_idx_in_pattern": 0,
        "candidates": [
            {"edit_type": "remove_atom",
             "remove_idx_in_pattern": 1,
             "center_idx_in_pattern": 0,
             "name": "amine (NCO hydrolyzed)",
             "rationale": "이소시아네이트(R-N=C=O)는 매우 반응성이 높은 친전자체로, "
                          "단백질/아미노기와 쉽게 부가반응을 일으켜 직업성 천식·과민증을 "
                          "유발하는 것으로 잘 알려짐(TDI, MDI 등 산업용 이소시아네이트 "
                          "사례). 체내/환경에서 실제로 일어나는 가수분해 경로(R-NCO + H2O "
                          "-> R-NH2 + CO2)와 동일하게 카르보닐 탄소와 산소를 제거하고 "
                          "질소만 남겨 아민으로 전환 (검증 필요)"},
        ],
    },
    "triple_bond": {
        "problem_smarts": "C#C",
        "edit_method": "atom_edit",
        "target_idx_pair_in_pattern": (0, 1),
        "candidates": [
            {"edit_type": "reduce_bond", "name": "alkene (partially reduced)",
             "rationale": "말단 알카인(삼중결합)은 CYP450 효소에 의해 기계기반 억제"
                          "(mechanism-based inhibition) 경로로 대사되며, 반응성 케텐/"
                          "에폭사이드 중간체를 형성해 효소를 비가역적으로 불활성화할 "
                          "수 있음(에티닐에스트라디올 등에서 알려진 메커니즘). 삼중결합을 "
                          "이중결합으로 환원하여 반응성을 낮춤 (검증 필요, 완전 포화가 "
                          "아닌 부분 환원)"},
        ],
    },
    "stilbene": {
        "problem_smarts": "c-[CX3]=[CX3]-c",
        "edit_method": "atom_edit",
        "target_idx_pair_in_pattern": (1, 2),
        "candidates": [
            {"edit_type": "reduce_bond", "name": "diarylethane (reduced)",
             "rationale": "스틸벤 구조(두 방향족 고리를 잇는 C=C)는 디에틸스틸베스트롤"
                          "(DES)처럼 내분비교란 및 대사 산화를 통한 반응성 중간체 형성이 "
                          "알려진 골격. 이중결합을 환원하여 평면성을 낮추고 대사 반응성을 "
                          "완화함 (검증 필요, 에스트로겐 수용체 결합에 필요한 형태 자체를 "
                          "훼손할 수 있어 신중한 해석 필요)"},
        ],
    },
    "beta-keto/anhydride": {
        "edit_method": "atom_edit",
        "problem_smarts": "C(=O)OC(=O)",
        "center_idx_in_pattern": 2,
        "candidates": [
            {"edit_type": "remove_atom",
             "remove_idx_in_pattern": 3,
             "center_idx_in_pattern": 2,
             "name": "carboxylic acid (anhydride hydrolyzed)",
             "rationale": "산 무수물(R-C(=O)-O-C(=O)-R')은 강한 아실화제로 단백질 아미노산 "
                          "잔기와 쉽게 반응하며, 수용액 환경에서 자발적으로 가수분해되어 "
                          "두 개의 카르복실산으로 분해되는 것이 자연스러운 무독화 경로임. "
                          "한쪽 아실기를 제거하여 이 가수분해 최종형(카르복실산)으로 직접 "
                          "전환 (검증 필요). ※ 대안 후보(무수물->아마이드/이미드 bioisostere) "
                          "는 문헌 확인 후 추가 예정"},
        ],
    },
}

def get_replacement_candidates(rule_name: str) -> dict | None:
    """rule_name에 해당하는 치환 정보(SMARTS + 후보 리스트)를 반환. 없으면 None."""
    return REPLACEMENT_LIBRARY.get(rule_name)

Overwriting src/tools/replacement_library.py


In [25]:
importlib.reload(src.tools.replacement_library)
importlib.reload(src.tools.atom_editor)
importlib.reload(src.tools.toxicophore_detector)
importlib.reload(src.tools.molecule_editor)
from src.tools.toxicophore_detector import detect_toxicophores
from src.tools.molecule_editor import propose_fix, iterative_fix_loop

print(list(get_replacement_candidates.__globals__['REPLACEMENT_LIBRARY'].keys()))

result_full_chain3 = iterative_fix_loop("CC(C)OC(=S)[S-]", max_iterations=10)
print("\n디티오카바메이트 전체 연쇄:", result_full_chain3['status'])
for h in result_full_chain3['history']:
    print(h)

print("\n=== 회귀 테스트 ===")
print(propose_fix("O=C(O)CCl", "alkyl_halide", candidate_idx=0))
print(propose_fix("CCCCCCCCOS(=O)(=O)[O-]", "sulphate", candidate_idx=0))

['nitro_group', 'aldehyde', 'Michael_acceptor_1', 'acid_halide', 'alkyl_halide', 'aniline', 'Sulfonic_acid_2', 'imine_1_oxime', 'imine_1_general', 'catechol', 'Thiocarbonyl_group', 'thiol_2', 'thiol_1_dithiocarbamate', 'thiol_1_thiocarboxylate', 'het-C-het_not_in_ring', 'hydroquinone', 'azo_A(324)', 'Three-membered_heterocycle', 'diketo_group', 'thioester', 'N-nitroso', 'hydrazine', 'sulphate', 'N_oxide', '2-halo_pyridine', 'disulphide', 'quinone_A(370)', 'isocyanate', 'triple_bond', 'stilbene', 'beta-keto/anhydride']

디티오카바메이트 전체 연쇄: stuck
{'step': 0, 'smiles': 'CC(C)OC(=S)[S-]', 'problems': [{'rule_name': 'Thiocarbonyl_group', 'atom_indices': [4, 5]}, {'rule_name': 'thiol_1', 'atom_indices': [6]}]}
{'step': 1, 'smiles': 'CC(C)OC(=O)[S-]', 'fixed_rule': 'Thiocarbonyl_group', 'problem_reason': '규칙 기반(리스트 순서대로)', 'candidate_used': 'carbonyl (O replacing S)', 'candidate_reason': '규칙 기반(고정 인덱스)', 'problems': [{'rule_name': 'thioester', 'atom_indices': [4, 5, 6]}, {'rule_name': 'thiol_1', 

In [26]:
%%writefile src/tools/toxicophore_detector.py
from rdkit import Chem
from rdkit.Chem import FilterCatalog

def _build_catalog():
    params = FilterCatalog.FilterCatalogParams()
    params.AddCatalog(FilterCatalog.FilterCatalogParams.FilterCatalogs.PAINS)
    params.AddCatalog(FilterCatalog.FilterCatalogParams.FilterCatalogs.BRENK)
    return FilterCatalog.FilterCatalog(params)

_catalog = _build_catalog()
_oxime_pattern = Chem.MolFromSmarts("C=N[OX2H1]")
_guanidine_pattern = Chem.MolFromSmarts("[$(C(N)(N)=N)]")

_DUPLICATE_RULE_MAP = {
    "catechol_A(92)": "catechol",
    "diazo_group": "azo_A(324)",
    "oxime_1": "imine_1_oxime",
    "chinone_1": "quinone_A(370)",
}


def _refine_imine1(mol, atom_indices):
    """imine_1은 옥심(C=N-OH), 구아니딘(N-C(=N)-N), 일반 이민(C=N-R)을
    모두 포함하는 넓은 카테고리이므로, 실제 매치 부분의 화학적 맥락을
    확인해 이름을 세분화한다."""
    if mol.HasSubstructMatch(_oxime_pattern):
        matches = mol.GetSubstructMatches(_oxime_pattern)
        for match in matches:
            if set(match) & set(atom_indices):
                return "imine_1_oxime"
    if mol.HasSubstructMatch(_guanidine_pattern):
        matches = mol.GetSubstructMatches(_guanidine_pattern)
        for match in matches:
            if set(match) & set(atom_indices):
                return "imine_1_guanidine"
    return "imine_1_general"


def _refine_thiol1(mol, atom_indices):
    """thiol_1은 FilterCatalog 원본이 음이온 황 원자 1개만 매치하는 넓은
    규칙이므로, 그 황이 붙은 탄소의 나머지 결합을 확인해 세분화한다:
    이웃 탄소가 C=S도 가지면 디티오카바메이트, C=O를 가지면
    티오카르복실산염, 둘 다 아니면 일반형(미지원)으로 분류한다."""
    s_idx = atom_indices[0]
    s_atom = mol.GetAtomWithIdx(s_idx)
    for nbr in s_atom.GetNeighbors():
        for nbr2 in nbr.GetNeighbors():
            if nbr2.GetIdx() == s_idx:
                continue
            bond2 = mol.GetBondBetweenAtoms(nbr.GetIdx(), nbr2.GetIdx())
            if bond2 is None or bond2.GetBondTypeAsDouble() != 2.0:
                continue
            if nbr2.GetSymbol() == 'S':
                return "thiol_1_dithiocarbamate"
            if nbr2.GetSymbol() == 'O':
                return "thiol_1_thiocarboxylate"
    return "thiol_1_general"


def detect_toxicophores(smiles: str) -> list[dict]:
    """
    분자의 SMILES를 받아, FilterCatalog(PAINS+BRENK)에 매치되는
    문제 구조(toxicophore)들을 찾아서 규칙 이름과 해당 원자 인덱스를 반환.
    imine_1은 옥심/구아니딘/일반이민, thiol_1은 디티오카바메이트/
    티오카르복실산염/일반형 하위형으로 세분화하여 반환한다.
    aniline은 FilterCatalog의 단순 [NH2] 탐지 대신, replacement_library의
    확장된 패턴(para-치환 벤젠 포함)을 그대로 사용해 재정의한다.
    PAINS/BRENK가 동일하거나 부분적으로 겹치는 구조를 서로 다른 이름/원자
    범위로 중복 보고하는 경우, 같은 rule_name에 원자 인덱스가 하나라도
    겹치면 중복으로 간주해 제거한다(완전히 동일한 인덱스일 필요는 없음).
    imine_1_general과 isocyanate가 같은 원자(누적이중결합)를 가리키면
    처리 가능한 isocyanate를 우선하고 imine_1_general은 제거한다.
    """
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return []

    results = []

    for entry in _catalog.GetMatches(mol):
        for fm in entry.GetFilterMatches(mol):
            atom_indices = sorted(set(mol_idx for _, mol_idx in fm.atomPairs))
            rule_name = entry.GetDescription()

            if rule_name == "imine_1":
                rule_name = _refine_imine1(mol, atom_indices)
            elif rule_name == "thiol_1":
                rule_name = _refine_thiol1(mol, atom_indices)
            elif rule_name == "aniline":
                continue
            elif rule_name in _DUPLICATE_RULE_MAP:
                rule_name = _DUPLICATE_RULE_MAP[rule_name]

            is_duplicate = any(
                r['rule_name'] == rule_name and set(r['atom_indices']) & set(atom_indices)
                for r in results
            )
            if is_duplicate:
                continue

            results.append({
                "rule_name": rule_name,
                "atom_indices": atom_indices,
            })

    from src.tools.replacement_library import get_replacement_candidates
    aniline_info = get_replacement_candidates("aniline")
    if aniline_info:
        aniline_pattern = Chem.MolFromSmarts(aniline_info["problem_smarts"])
        if mol.HasSubstructMatch(aniline_pattern):
            matches = mol.GetSubstructMatches(aniline_pattern)
            for match in matches:
                atom_indices = sorted(set(match))
                results.append({
                    "rule_name": "aniline",
                    "atom_indices": atom_indices,
                })

    isocyanate_atom_sets = [set(r['atom_indices']) for r in results if r['rule_name'] == 'isocyanate']
    if isocyanate_atom_sets:
        results = [
            r for r in results
            if not (r['rule_name'] == 'imine_1_general'
                    and any(set(r['atom_indices']) & iso_set for iso_set in isocyanate_atom_sets))
        ]

    return results

Overwriting src/tools/toxicophore_detector.py


In [27]:
importlib.reload(src.tools.toxicophore_detector)
importlib.reload(src.tools.molecule_editor)
from src.tools.toxicophore_detector import detect_toxicophores
from src.tools.molecule_editor import iterative_fix_loop

result_full_chain4 = iterative_fix_loop("CC(C)OC(=S)[S-]", max_iterations=10)
print("상태:", result_full_chain4['status'])
for h in result_full_chain4['history']:
    print(h)

상태: stuck
{'step': 0, 'smiles': 'CC(C)OC(=S)[S-]', 'problems': [{'rule_name': 'Thiocarbonyl_group', 'atom_indices': [4, 5]}, {'rule_name': 'thiol_1_dithiocarbamate', 'atom_indices': [6]}]}
{'step': 1, 'smiles': 'CC(C)OC(=O)[S-]', 'fixed_rule': 'Thiocarbonyl_group', 'problem_reason': '규칙 기반(리스트 순서대로)', 'candidate_used': 'carbonyl (O replacing S)', 'candidate_reason': '규칙 기반(고정 인덱스)', 'problems': [{'rule_name': 'thioester', 'atom_indices': [4, 5, 6]}, {'rule_name': 'thiol_1_thiocarboxylate', 'atom_indices': [6]}]}


In [28]:
print("thioester:", propose_fix("CC(C)OC(=O)[S-]", "thioester", candidate_idx=0))
print("thiol_1_thiocarboxylate:", propose_fix("CC(C)OC(=O)[S-]", "thiol_1_thiocarboxylate", candidate_idx=0))

thioester: None
thiol_1_thiocarboxylate: {'new_smiles': 'CC(C)OC(=O)[O-]', 'candidate_used': 'carboxylate (O replacing S)', 'rationale': '티오카르복실산 음이온(R-C(=O)-S-)의 황을 산소로 대체하여 카르복실산염(R-C(=O)-O-)으로 전환. 황 원자의 금속 킬레이팅 및 친핵성 반응성을 제거함 (검증 필요)', 'is_valid': True}


In [29]:
problems_now = detect_toxicophores("CC(C)OC(=O)[S-]")
print(problems_now)
known_now = [p for p in problems_now if get_replacement_candidates(p['rule_name']) is not None]
print("known:", [p['rule_name'] for p in known_now])

[{'rule_name': 'thioester', 'atom_indices': [4, 5, 6]}, {'rule_name': 'thiol_1_thiocarboxylate', 'atom_indices': [6]}]
known: ['thioester', 'thiol_1_thiocarboxylate']


In [31]:
import inspect
print(inspect.getsource(iterative_fix_loop))
import importlib
import src.tools.molecule_editor
importlib.reload(src.tools.molecule_editor)
from src.tools.molecule_editor import iterative_fix_loop, propose_fix, canonicalize

print(inspect.getsource(iterative_fix_loop))

def iterative_fix_loop(smiles: str, max_iterations: int = 10, candidate_idx: int = 0,
                        llm_client=None, llm_model=None, llm_client_type="gemini"):
    """진단->치환->재평가를 반복.
    llm_client가 주어지면: 어떤 문제부터 고칠지 + 어떤 후보를 쓸지 둘 다 LLM이 판단.
    llm_client_type: "gemini" 또는 "openai_compatible".
    llm_client가 없으면: 리스트 순서(known_problems[0]) + candidate_idx 고정값 사용.
    skipped_details/reason_detail: 연구자가 no_known_fix/stuck 사유를 바로
    확인할 수 있도록 사람이 읽을 수 있는 설명과 매치된 원자 정보를 함께 제공."""
    from src.tools.toxicophore_detector import detect_toxicophores
    from src.tools.agent import ask_llm_which_problem_to_fix, ask_llm_which_candidate_to_use

    current = canonicalize(smiles)
    seen = {current}
    history = [{"step": 0, "smiles": current}]
    skipped_rules = []
    skipped_details = []

    for step in range(1, max_iterations + 1):
        problems = detect_toxicophores(current)
        history[-1]["problems"] = problems

        if not problems:
            return {"status":

In [32]:
%%writefile src/tools/molecule_editor.py
from rdkit import Chem
from rdkit.Chem import rdMMPA
from src.tools.replacement_library import get_replacement_candidates


def _check_and_match(part_smiles, problem_pattern, pattern_size):
    """조각이 problem_pattern과 정확한 크기로 매치되는지 확인."""
    part_mol = Chem.MolFromSmiles(part_smiles.replace('[*:1]', 'C').replace('[*:2]', 'C'))
    if part_mol is None or not part_mol.HasSubstructMatch(problem_pattern):
        return False
    n_attachment = part_smiles.count('[*:')
    return part_mol.GetNumHeavyAtoms() - n_attachment == pattern_size


def find_core_and_target(smiles: str, rule_name: str):
    info = get_replacement_candidates(rule_name)
    if info is None:
        return None

    problem_pattern = Chem.MolFromSmarts(info['problem_smarts'])
    pattern_size = problem_pattern.GetNumAtoms()

    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None

    fragments1 = rdMMPA.FragmentMol(mol, maxCuts=1, resultsAsMols=False)
    for core, chain in fragments1:
        if core:
            continue
        parts = chain.split('.')
        if len(parts) != 2:
            continue
        for i, part in enumerate(parts):
            if _check_and_match(part, problem_pattern, pattern_size):
                return {"core": parts[1 - i], "target_removed": part}

    fragments2 = rdMMPA.FragmentMol(mol, maxCuts=2, resultsAsMols=False)
    for core, chain in fragments2:
        if not core:
            continue
        chain_parts = chain.split('.')
        if len(chain_parts) != 2:
            continue
        for i, part in enumerate(chain_parts):
            if not _check_and_match(part, problem_pattern, pattern_size):
                continue
            other_chain_part = chain_parts[1 - i]

            target_ap = '[*:1]' if '[*:1]' in part else ('[*:2]' if '[*:2]' in part else None)
            if target_ap is None:
                continue

            core_mol = Chem.MolFromSmiles(core)
            other_mol = Chem.MolFromSmiles(other_chain_part)
            if core_mol is None or other_mol is None:
                continue
            try:
                merged = Chem.molzip(core_mol, other_mol)
            except Exception:
                continue

            merged_smiles = Chem.MolToSmiles(merged)
            if merged_smiles.count('[*:') != 1:
                continue
            if '[*:1]' not in merged_smiles:
                merged_smiles = merged_smiles.replace('[*:2]', '[*:1]')

            return {"core": merged_smiles, "target_removed": part}

    return None


def reassemble_molecule(core_smiles: str, rule_name: str, candidate_idx: int = 0):
    info = get_replacement_candidates(rule_name)
    if info is None or candidate_idx >= len(info['candidates']):
        return None
    candidate = info['candidates'][candidate_idx]

    core_mol = Chem.MolFromSmiles(core_smiles)
    replacement_mol = Chem.MolFromSmiles(f"[*:1]{candidate['smiles']}")
    if core_mol is None or replacement_mol is None:
        return None

    try:
        combined = Chem.molzip(core_mol, replacement_mol)
        new_smiles = Chem.MolToSmiles(combined)
    except Exception:
        return None

    is_valid = Chem.MolFromSmiles(new_smiles) is not None

    return {
        "new_smiles": new_smiles,
        "candidate_used": candidate['name'],
        "rationale": candidate['rationale'],
        "is_valid": is_valid,
    }


def propose_fix(smiles: str, rule_name: str, candidate_idx: int = 0):
    info = get_replacement_candidates(rule_name)
    if info is None:
        return None

    if info.get("edit_method") == "atom_edit":
        from src.tools.atom_editor import apply_atom_edit_from_rule
        return apply_atom_edit_from_rule(smiles, rule_name, candidate_idx)

    located = find_core_and_target(smiles, rule_name)
    if located is None:
        return None
    return reassemble_molecule(located['core'], rule_name, candidate_idx)


def canonicalize(smiles: str):
    mol = Chem.MolFromSmiles(smiles)
    return Chem.MolToSmiles(mol) if mol else None


def iterative_fix_loop(smiles: str, max_iterations: int = 10, candidate_idx: int = 0,
                        llm_client=None, llm_model=None, llm_client_type="gemini"):
    """진단->치환->재평가를 반복. known 규칙 중 우선순위가 가장 높은 것이
    propose_fix에서 실패하면, stuck 처리 전에 같은 분자의 다른 known
    규칙들을 순서대로 시도한다."""
    from src.tools.toxicophore_detector import detect_toxicophores
    from src.tools.agent import ask_llm_which_problem_to_fix, ask_llm_which_candidate_to_use

    current = canonicalize(smiles)
    seen = {current}
    history = [{"step": 0, "smiles": current}]
    skipped_rules = []
    skipped_details = []
    flagged_for_review = set()

    for step in range(1, max_iterations + 1):
        problems = detect_toxicophores(current)
        history[-1]["problems"] = problems

        if not problems:
            return {"status": "success", "final_smiles": current, "history": history,
                    "skipped_rules": skipped_rules, "skipped_details": skipped_details}

        known_problems = [p for p in problems if get_replacement_candidates(p['rule_name']) is not None
                          and p['rule_name'] not in flagged_for_review]
        unknown_problems = [p for p in problems if get_replacement_candidates(p['rule_name']) is None]

        for p in unknown_problems:
            if p['rule_name'] not in skipped_rules:
                skipped_rules.append(p['rule_name'])
                mol_cur = Chem.MolFromSmiles(current)
                matched_atoms = p['atom_indices']
                atom_symbols = [mol_cur.GetAtomWithIdx(i).GetSymbol() for i in matched_atoms] if mol_cur else []
                skipped_details.append({
                    "rule_name": p['rule_name'],
                    "reason": f"라이브러리에 등록되지 않은 규칙입니다. FilterCatalog(PAINS/BRENK)가 "
                              f"'{p['rule_name']}'로 진단했으며, 매치된 원자 인덱스는 {matched_atoms}"
                              f"(원소: {atom_symbols})입니다. 이 구조에 대한 치환 규칙을 "
                              f"replacement_library.py에 추가하면 자동으로 처리 가능합니다.",
                    "atom_indices": matched_atoms,
                })

        if not known_problems:
            return {"status": "no_known_fix", "final_smiles": current, "history": history,
                    "skipped_rules": skipped_rules, "skipped_details": skipped_details}

        if llm_client is not None:
            problem_decision = ask_llm_which_problem_to_fix(llm_client, llm_model, current, problems, client_type=llm_client_type)
            preferred_rule = problem_decision['rule_name']
            problem_reason = problem_decision.get('reason', '')
            ordered_rules = [preferred_rule] + [p['rule_name'] for p in known_problems if p['rule_name'] != preferred_rule]
        else:
            problem_reason = "규칙 기반(리스트 순서대로)"
            ordered_rules = [p['rule_name'] for p in known_problems]

        fixed = None
        target_rule = None
        candidate_reason = None
        failed_attempts = []

        for candidate_rule in ordered_rules:
            if llm_client is not None:
                candidate_decision = ask_llm_which_candidate_to_use(llm_client, llm_model, current, candidate_rule, client_type=llm_client_type)
                chosen_candidate_idx = candidate_decision['candidate_idx']
                this_candidate_reason = candidate_decision.get('reason', '')

                if chosen_candidate_idx == -1:
                    flagged_for_review.add(candidate_rule)
                    if candidate_rule not in skipped_rules:
                        skipped_rules.append(candidate_rule)
                    skipped_details.append({
                        "rule_name": candidate_rule,
                        "reason": f"LLM이 치환을 보류했습니다: {this_candidate_reason} "
                                  f"(이 분자가 [참고] 사항에 해당하는 안전한 실사용 사례와 유사하다고 "
                                  f"판단되어, 자동 치환 대신 연구자의 직접 검토를 권장합니다.)",
                        "atom_indices": next((p['atom_indices'] for p in problems if p['rule_name'] == candidate_rule), []),
                    })
                    continue
            else:
                chosen_candidate_idx = candidate_idx
                this_candidate_reason = "규칙 기반(고정 인덱스)"

            attempt = propose_fix(current, candidate_rule, chosen_candidate_idx)
            if attempt is not None and attempt.get('is_valid'):
                fixed = attempt
                target_rule = candidate_rule
                candidate_reason = this_candidate_reason
                break
            else:
                failed_attempts.append(candidate_rule)

        if fixed is None:
            reason_detail = (f"이 단계에서 known 규칙 {failed_attempts} 전부를 순서대로 시도했으나 "
                              f"모두 실행에 실패했습니다. 흔한 원인: 유기금속/무기염 등 특수 화학종, "
                              f"고리 구조와의 예상치 못한 충돌, 또는 원자가 계산 오류입니다.")
            return {"status": "stuck", "reason": f"시도한 규칙 {failed_attempts} 모두 치환 실패",
                    "reason_detail": reason_detail,
                    "final_smiles": current, "history": history,
                    "skipped_rules": skipped_rules, "skipped_details": skipped_details}

        new_current = canonicalize(fixed['new_smiles'])

        if new_current in seen:
            return {"status": "cycle_detected", "final_smiles": current, "history": history,
                    "skipped_rules": skipped_rules, "skipped_details": skipped_details}

        seen.add(new_current)
        current = new_current
        history.append({
            "step": step,
            "smiles": current,
            "fixed_rule": target_rule,
            "problem_reason": problem_reason,
            "candidate_used": fixed['candidate_used'],
            "candidate_reason": candidate_reason,
        })

    return {"status": "max_iterations_reached", "final_smiles": current, "history": history,
            "skipped_rules": skipped_rules, "skipped_details": skipped_details}

Overwriting src/tools/molecule_editor.py


In [33]:
importlib.reload(src.tools.molecule_editor)
import src.tools.molecule_editor
from src.tools.molecule_editor import iterative_fix_loop
import inspect
print(inspect.getsource(iterative_fix_loop))

def iterative_fix_loop(smiles: str, max_iterations: int = 10, candidate_idx: int = 0,
                        llm_client=None, llm_model=None, llm_client_type="gemini"):
    """진단->치환->재평가를 반복. known 규칙 중 우선순위가 가장 높은 것이
    propose_fix에서 실패하면, stuck 처리 전에 같은 분자의 다른 known
    규칙들을 순서대로 시도한다."""
    from src.tools.toxicophore_detector import detect_toxicophores
    from src.tools.agent import ask_llm_which_problem_to_fix, ask_llm_which_candidate_to_use

    current = canonicalize(smiles)
    seen = {current}
    history = [{"step": 0, "smiles": current}]
    skipped_rules = []
    skipped_details = []
    flagged_for_review = set()

    for step in range(1, max_iterations + 1):
        problems = detect_toxicophores(current)
        history[-1]["problems"] = problems

        if not problems:
            return {"status": "success", "final_smiles": current, "history": history,
                    "skipped_rules": skipped_rules, "skipped_details": skipped_details}

        known_problems 

In [34]:
result_final_test = iterative_fix_loop("CC(C)OC(=S)[S-]", max_iterations=10)
print("상태:", result_final_test['status'])
for h in result_final_test['history']:
    print(h)

상태: success
{'step': 0, 'smiles': 'CC(C)OC(=S)[S-]', 'problems': [{'rule_name': 'Thiocarbonyl_group', 'atom_indices': [4, 5]}, {'rule_name': 'thiol_1_dithiocarbamate', 'atom_indices': [6]}]}
{'step': 1, 'smiles': 'CC(C)OC(=O)[S-]', 'fixed_rule': 'Thiocarbonyl_group', 'problem_reason': '규칙 기반(리스트 순서대로)', 'candidate_used': 'carbonyl (O replacing S)', 'candidate_reason': '규칙 기반(고정 인덱스)', 'problems': [{'rule_name': 'thioester', 'atom_indices': [4, 5, 6]}, {'rule_name': 'thiol_1_thiocarboxylate', 'atom_indices': [6]}]}
{'step': 2, 'smiles': 'CC(C)OC(=O)[O-]', 'fixed_rule': 'thiol_1_thiocarboxylate', 'problem_reason': '규칙 기반(리스트 순서대로)', 'candidate_used': 'carboxylate (O replacing S)', 'candidate_reason': '규칙 기반(고정 인덱스)', 'problems': []}


In [35]:
!git add src/tools/molecule_editor.py src/tools/toxicophore_detector.py src/tools/replacement_library.py
!git status

On branch main
Your branch is up to date with 'origin/main'.

Changes to be committed:
  (use "git restore --staged <file>..." to unstage)
	modified:   src/tools/molecule_editor.py
	modified:   src/tools/replacement_library.py
	modified:   src/tools/toxicophore_detector.py



In [36]:
!git commit -m "Session fix batch: (1) iterative_fix_loop now retries all known rules in priority order before giving up (previously stuck immediately if the top-priority rule failed at execution, even when other known rules on the same molecule would have succeeded). (2) Split thiol_1 into thiol_1_dithiocarbamate and thiol_1_thiocarboxylate — FilterCatalog's thiol_1 only matches a single anionic sulfur atom and covers both chemotypes, which previously caused execution failures when the narrow dithiocarbamate SMARTS was applied to thiocarboxylate structures. (3) Narrowed Sulfonic_acid_2 SMARTS to require direct C-S bond, excluding sulfate esters (which belong to the separate 'sulphate' rule). Verified full chain: dithiocarbamate -> Thiocarbonyl_group fix -> thiocarboxylate (auto-detected) -> carboxylate fix -> success (3 steps)."
!git push origin main

[main 67a5698] Session fix batch: (1) iterative_fix_loop now retries all known rules in priority order before giving up (previously stuck immediately if the top-priority rule failed at execution, even when other known rules on the same molecule would have succeeded). (2) Split thiol_1 into thiol_1_dithiocarbamate and thiol_1_thiocarboxylate — FilterCatalog's thiol_1 only matches a single anionic sulfur atom and covers both chemotypes, which previously caused execution failures when the narrow dithiocarbamate SMARTS was applied to thiocarboxylate structures. (3) Narrowed Sulfonic_acid_2 SMARTS to require direct C-S bond, excluding sulfate esters (which belong to the separate 'sulphate' rule). Verified full chain: dithiocarbamate -> Thiocarbonyl_group fix -> thiocarboxylate (auto-detected) -> carboxylate fix -> success (3 steps).
 3 files changed, 131 insertions(+), 70 deletions(-)
Enumerating objects: 13, done.
Counting objects: 100% (13/13), done.
Delta compression using up to 2 thread